In [ ]:
# Consolidated experiment configuration for dev/full pipeline execution


from pathlib import Path
import json
import datetime
import platform
import random
import numpy as np
import torch


#master reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


#dataset roots

CELEBDF_ROOT = Path(r" Path to Celeb-DF-v2")
FFPP_ROOT = Path(r"Path to FaceForensicsPP")


#  project/output root

PROJECT_ROOT = Path(r"output path")

DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
METADATA_ROOT = DATA_ROOT / "metadata"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
LOG_ROOT = OUTPUT_ROOT / "logs"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
REPORT_ROOT = OUTPUT_ROOT / "reports"
RUN_ROOT = OUTPUT_ROOT / "run_artifacts"

for p in [DATA_ROOT, RAW_ROOT, PROCESSED_ROOT, METADATA_ROOT,
          OUTPUT_ROOT, CHECKPOINT_ROOT, LOG_ROOT, FIGURE_ROOT, REPORT_ROOT, RUN_ROOT]:
    p.mkdir(parents=True, exist_ok=True)


#  execution mode

RUN_MODE = "full"   # "dev" or "full"

# For dev mode
DEV_MAX_VIDEOS_PER_GROUP = 50

# For full mode
# Set to None to use all available videos
FULL_MAX_VIDEOS_PER_GROUP = None


#  split ratios

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15


#  preprocessing

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv", ".mpeg", ".mpg", ".m4v"}

FRAMES_PER_VIDEO = 8
OUTPUT_IMAGE_EXT = ".jpg"
JPEG_QUALITY = 95

FACE_IMAGE_SIZE = 224
FACE_CONF_THRESHOLD = 0.90
FACE_MARGIN_RATIO = 0.20
KEEP_ONLY_LARGEST_FACE = True


#  binary model training

BINARY_MODEL_NAME = "resnet18_video_meanpool"
BINARY_NUM_EPOCHS = 8
BINARY_LEARNING_RATE = 1e-4
BINARY_WEIGHT_DECAY = 1e-4
BINARY_USE_CLASS_WEIGHT = True
BINARY_FREEZE_BACKBONE = False
BINARY_MAX_GRAD_NORM = 1.0


#  binary calibration / uncertainty

BINARY_FORENSIC_THRESHOLD_RAW = 0.33
BINARY_FORENSIC_THRESHOLD_CALIBRATED = 0.48

BINARY_UNCERTAIN_LOW = 0.40
BINARY_UNCERTAIN_HIGH = 0.55


#  multiclass model training

MULTICLASS_MODEL_NAME = "resnet18_video_meanpool"
MULTICLASS_NUM_EPOCHS = 10
MULTICLASS_LEARNING_RATE = 1e-4
MULTICLASS_WEIGHT_DECAY = 1e-4
MULTICLASS_USE_CLASS_WEIGHT = True
MULTICLASS_FREEZE_BACKBONE = False
MULTICLASS_MAX_GRAD_NORM = 1.0

MULTICLASS_LABELS = {
    "real": 0,
    "Deepfakes": 1,
    "Face2Face": 2,
    "FaceShifter": 3,
    "FaceSwap": 4,
    "NeuralTextures": 5,
    "Celeb-synthesis": 6,
}


#  multiclass uncertainty

MULTICLASS_UNCERTAIN_PROB_THRESHOLD = 0.60
MULTICLASS_UNCERTAIN_MARGIN_THRESHOLD = 0.15


#  robustness protocol

ROBUSTNESS_DEGRADATIONS = [
    {"name": "clean", "type": "none", "severity": "none"},
    {"name": "jpeg_q50", "type": "jpeg", "quality": 50, "severity": "moderate"},
    {"name": "blur_r1.5", "type": "blur", "radius": 1.5, "severity": "moderate"},
    {"name": "resize_0.5x", "type": "resize", "scale": 0.5, "severity": "moderate"},
    {"name": "noise_std10", "type": "noise", "std": 10.0, "severity": "moderate"},
]


#  explainability selection

MAX_TEST_SAMPLES_TO_EXPLAIN = 8


# Runtime + config export

RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
EXPERIMENT_NAME = f"{RUN_MODE}_deepfake_forensics_pipeline_{RUN_TIMESTAMP}"
EXPERIMENT_RUN_DIR = RUN_ROOT / EXPERIMENT_NAME
EXPERIMENT_RUN_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_name": EXPERIMENT_NAME,
    "run_timestamp": RUN_TIMESTAMP,
    "run_mode": RUN_MODE,
    "seed": SEED,

    "paths": {
        "project_root": str(PROJECT_ROOT),
        "celebdf_root": str(CELEBDF_ROOT),
        "ffpp_root": str(FFPP_ROOT),
        "data_root": str(DATA_ROOT),
        "processed_root": str(PROCESSED_ROOT),
        "metadata_root": str(METADATA_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "checkpoint_root": str(CHECKPOINT_ROOT),
        "figure_root": str(FIGURE_ROOT),
        "report_root": str(REPORT_ROOT),
        "experiment_run_dir": str(EXPERIMENT_RUN_DIR),
    },

    "sampling": {
        "dev_max_videos_per_group": DEV_MAX_VIDEOS_PER_GROUP,
        "full_max_videos_per_group": FULL_MAX_VIDEOS_PER_GROUP,
    },

    "splits": {
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
    },

    "preprocessing": {
        "video_extensions": sorted(list(VIDEO_EXTENSIONS)),
        "frames_per_video": FRAMES_PER_VIDEO,
        "output_image_ext": OUTPUT_IMAGE_EXT,
        "jpeg_quality": JPEG_QUALITY,
        "face_image_size": FACE_IMAGE_SIZE,
        "face_conf_threshold": FACE_CONF_THRESHOLD,
        "face_margin_ratio": FACE_MARGIN_RATIO,
        "keep_only_largest_face": KEEP_ONLY_LARGEST_FACE,
    },

    "binary_model": {
        "model_name": BINARY_MODEL_NAME,
        "num_epochs": BINARY_NUM_EPOCHS,
        "learning_rate": BINARY_LEARNING_RATE,
        "weight_decay": BINARY_WEIGHT_DECAY,
        "use_class_weight": BINARY_USE_CLASS_WEIGHT,
        "freeze_backbone": BINARY_FREEZE_BACKBONE,
        "max_grad_norm": BINARY_MAX_GRAD_NORM,
        "forensic_threshold_raw": BINARY_FORENSIC_THRESHOLD_RAW,
        "forensic_threshold_calibrated": BINARY_FORENSIC_THRESHOLD_CALIBRATED,
        "uncertain_low": BINARY_UNCERTAIN_LOW,
        "uncertain_high": BINARY_UNCERTAIN_HIGH,
    },

    "multiclass_model": {
        "model_name": MULTICLASS_MODEL_NAME,
        "num_epochs": MULTICLASS_NUM_EPOCHS,
        "learning_rate": MULTICLASS_LEARNING_RATE,
        "weight_decay": MULTICLASS_WEIGHT_DECAY,
        "use_class_weight": MULTICLASS_USE_CLASS_WEIGHT,
        "freeze_backbone": MULTICLASS_FREEZE_BACKBONE,
        "max_grad_norm": MULTICLASS_MAX_GRAD_NORM,
        "labels": MULTICLASS_LABELS,
        "uncertain_prob_threshold": MULTICLASS_UNCERTAIN_PROB_THRESHOLD,
        "uncertain_margin_threshold": MULTICLASS_UNCERTAIN_MARGIN_THRESHOLD,
    },

    "robustness": {
        "degradations": ROBUSTNESS_DEGRADATIONS,
    },

    "explainability": {
        "max_test_samples_to_explain": MAX_TEST_SAMPLES_TO_EXPLAIN,
    },

    "environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    },
}

CONFIG_JSON_PATH = EXPERIMENT_RUN_DIR / "experiment_config.json"
with open(CONFIG_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print("Experiment name        :", EXPERIMENT_NAME)
print("Run mode               :", RUN_MODE)
print("Config JSON            :", CONFIG_JSON_PATH)
print("CUDA available         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device            :", torch.cuda.get_device_name(0))

print("\nKey settings:")
print("  Frames per video     :", FRAMES_PER_VIDEO)
print("  Face size            :", FACE_IMAGE_SIZE)
print("  Binary epochs        :", BINARY_NUM_EPOCHS)
print("  Multiclass epochs    :", MULTICLASS_NUM_EPOCHS)
print("  Full max/group       :", FULL_MAX_VIDEOS_PER_GROUP)
print("  Dev max/group        :", DEV_MAX_VIDEOS_PER_GROUP)

In [ ]:

# Full-run metadata generation from consolidated config


from pathlib import Path
import hashlib
import json
import pandas as pd


# This cell assumes already been run and created:
# - CONFIG
# - CELEBDF_ROOT
# - FFPP_ROOT
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - RUN_MODE
# - DEV_MAX_VIDEOS_PER_GROUP
# - FULL_MAX_VIDEOS_PER_GROUP
# - VIDEO_EXTENSIONS
# - SEED



# Label definitions

BINARY_LABELS = {
    "real": 0,
    "fake": 1,
}

FAMILY_LABELS = {
    "real": 0,
    "Deepfakes": 1,
    "Face2Face": 2,
    "FaceShifter": 3,
    "FaceSwap": 4,
    "NeuralTextures": 5,
    "Celeb-synthesis": 6,
}

REAL_SUBSETS_CELEBDF = {"Celeb-real", "YouTube-real"}
FAKE_SUBSETS_CELEBDF = {"Celeb-synthesis"}

REAL_SUBSETS_FFPP = {"youtube"}
FAKE_SUBSETS_FFPP = {"Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"}


# CHANGE THIS IF NEEDED
# Output file names for full run

METADATA_ALL_CSV_FULL = METADATA_ROOT / "metadata_all_videos_fullrun.csv"
METADATA_SELECTED_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun.csv"
METADATA_SUMMARY_JSON_FULL = EXPERIMENT_RUN_DIR / "metadata_fullrun_summary.json"
CELEBDF_UNMATCHED_JSON = EXPERIMENT_RUN_DIR / "celebdf_unmatched_paths_fullrun.json"
FFPP_UNMATCHED_JSON = EXPERIMENT_RUN_DIR / "ffpp_unmatched_paths_fullrun.json"


# Helpers

def is_video_file(path: Path, allowed_exts=VIDEO_EXTENSIONS) -> bool:
    return path.is_file() and path.suffix.lower() in allowed_exts

def make_sample_id(dataset_name: str, relative_path: str) -> str:
    base = f"{dataset_name}::{relative_path}".encode("utf-8")
    return hashlib.md5(base).hexdigest()[:16]

def scan_videos_recursive(root_dir: Path):
    if not root_dir.exists():
        return []
    return sorted([p for p in root_dir.rglob("*") if is_video_file(p)])

def infer_celebdf_record(video_path: Path, dataset_root: Path):
    rel_path = video_path.relative_to(dataset_root)
    parts = rel_path.parts

    if len(parts) < 2:
        return None

    top_subset = parts[0]

    if top_subset in REAL_SUBSETS_CELEBDF:
        binary_name = "real"
        family_name = "real"
        is_real = 1
        is_fake = 0
    elif top_subset in FAKE_SUBSETS_CELEBDF:
        binary_name = "fake"
        family_name = "Celeb-synthesis"
        is_real = 0
        is_fake = 1
    else:
        return None

    relative_path_str = str(rel_path).replace("\\", "/")

    return {
        "sample_id": make_sample_id("CelebDF", relative_path_str),
        "dataset": "CelebDF",
        "source_subset": top_subset,
        "file_path": str(video_path.resolve()),
        "relative_path": relative_path_str,
        "file_name": video_path.name,
        "stem": video_path.stem,
        "file_ext": video_path.suffix.lower(),
        "binary_label": BINARY_LABELS[binary_name],
        "binary_label_name": binary_name,
        "family_label": FAMILY_LABELS[family_name],
        "family_label_name": family_name,
        "is_real": is_real,
        "is_fake": is_fake,
    }

def infer_ffpp_record(video_path: Path, dataset_root: Path):
    rel_path = video_path.relative_to(dataset_root)
    parts = rel_path.parts

    if len(parts) < 3:
        return None

    level0 = parts[0]
    level1 = parts[1]

    if level0 == "original_sequences" and level1 in REAL_SUBSETS_FFPP:
        subset_name = level1
        binary_name = "real"
        family_name = "real"
        is_real = 1
        is_fake = 0
    elif level0 == "manipulated_sequences" and level1 in FAKE_SUBSETS_FFPP:
        subset_name = level1
        binary_name = "fake"
        family_name = level1
        is_real = 0
        is_fake = 1
    else:
        return None

    relative_path_str = str(rel_path).replace("\\", "/")

    return {
        "sample_id": make_sample_id("FaceForensicsPP", relative_path_str),
        "dataset": "FaceForensicsPP",
        "source_subset": subset_name,
        "file_path": str(video_path.resolve()),
        "relative_path": relative_path_str,
        "file_name": video_path.name,
        "stem": video_path.stem,
        "file_ext": video_path.suffix.lower(),
        "binary_label": BINARY_LABELS[binary_name],
        "binary_label_name": binary_name,
        "family_label": FAMILY_LABELS[family_name],
        "family_label_name": family_name,
        "is_real": is_real,
        "is_fake": is_fake,
    }

def sample_by_mode(df):
    """
    In dev mode:
        cap by (dataset, family_label_name)
    In full mode:
        use all rows unless FULL_MAX_VIDEOS_PER_GROUP is set
    """
    if RUN_MODE == "dev":
        max_per_group = DEV_MAX_VIDEOS_PER_GROUP
    else:
        max_per_group = FULL_MAX_VIDEOS_PER_GROUP

    if max_per_group is None:
        out = df.copy().reset_index(drop=True)
        out["selected_for_run"] = 1
        return out

    sampled_parts = []
    grouped = df.groupby(["dataset", "family_label_name"], group_keys=False)

    for (dataset_name, family_name), group_df in grouped:
        group_df = group_df.copy().sample(frac=1.0, random_state=SEED)
        group_df = group_df.head(max_per_group)
        sampled_parts.append(group_df)

    out = pd.concat(sampled_parts, axis=0).reset_index(drop=True)
    out["selected_for_run"] = 1
    return out


# Scan datasets

celebdf_video_paths = scan_videos_recursive(CELEBDF_ROOT)
ffpp_video_paths = scan_videos_recursive(FFPP_ROOT)

print("Discovered video files:")
print("  CelebDF         :", len(celebdf_video_paths))
print("  FaceForensics++ :", len(ffpp_video_paths))


# Build metadata records

records = []

celebdf_unmatched = []
for p in celebdf_video_paths:
    rec = infer_celebdf_record(p, CELEBDF_ROOT)
    if rec is None:
        celebdf_unmatched.append(str(p))
    else:
        records.append(rec)

ffpp_unmatched = []
for p in ffpp_video_paths:
    rec = infer_ffpp_record(p, FFPP_ROOT)
    if rec is None:
        ffpp_unmatched.append(str(p))
    else:
        records.append(rec)

metadata_all_df = pd.DataFrame(records)

if len(metadata_all_df) == 0:
    raise RuntimeError("No metadata records were created. Check dataset folder structure.")

metadata_all_df = metadata_all_df.drop_duplicates(subset=["sample_id"]).copy()
metadata_all_df = metadata_all_df.sort_values(
    by=["dataset", "source_subset", "relative_path"]
).reset_index(drop=True)

metadata_selected_df = sample_by_mode(metadata_all_df)


# Save files

metadata_all_df.to_csv(METADATA_ALL_CSV_FULL, index=False)
metadata_selected_df.to_csv(METADATA_SELECTED_CSV_FULL, index=False)

with open(CELEBDF_UNMATCHED_JSON, "w", encoding="utf-8") as f:
    json.dump(celebdf_unmatched, f, indent=2)

with open(FFPP_UNMATCHED_JSON, "w", encoding="utf-8") as f:
    json.dump(ffpp_unmatched, f, indent=2)

summary = {
    "run_mode": RUN_MODE,
    "n_celebdf_discovered": int(len(celebdf_video_paths)),
    "n_ffpp_discovered": int(len(ffpp_video_paths)),
    "n_celebdf_unmatched": int(len(celebdf_unmatched)),
    "n_ffpp_unmatched": int(len(ffpp_unmatched)),
    "n_all_videos": int(len(metadata_all_df)),
    "n_selected_videos": int(len(metadata_selected_df)),
    "dataset_distribution_all": metadata_all_df["dataset"].value_counts().to_dict(),
    "family_distribution_all": metadata_all_df["family_label_name"].value_counts().to_dict(),
    "dataset_family_distribution_all": {
        f"{k[0]}__{k[1]}": int(v)
        for k, v in metadata_all_df.groupby(["dataset", "family_label_name"]).size().to_dict().items()
    },
    "dataset_distribution_selected": metadata_selected_df["dataset"].value_counts().to_dict(),
    "family_distribution_selected": metadata_selected_df["family_label_name"].value_counts().to_dict(),
    "dataset_family_distribution_selected": {
        f"{k[0]}__{k[1]}": int(v)
        for k, v in metadata_selected_df.groupby(["dataset", "family_label_name"]).size().to_dict().items()
    },
}

with open(METADATA_SUMMARY_JSON_FULL, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nSaved files:")
print("  All metadata CSV      :", METADATA_ALL_CSV_FULL)
print("  Selected metadata CSV :", METADATA_SELECTED_CSV_FULL)
print("  Summary JSON          :", METADATA_SUMMARY_JSON_FULL)

print("\nUnmatched path counts:")
print("  CelebDF unmatched     :", len(celebdf_unmatched))
print("  FF++ unmatched        :", len(ffpp_unmatched))

print("\nAll-video summary:")
print(
    metadata_all_df[["dataset", "source_subset", "binary_label_name", "family_label_name"]]
    .value_counts()
    .sort_index()
)

print("\nSelected-video summary:")
print(
    metadata_selected_df[["dataset", "source_subset", "binary_label_name", "family_label_name"]]
    .value_counts()
    .sort_index()
)

print("\nPreview of selected metadata:")
display(metadata_selected_df.head(10))

In [ ]:

# Full-run enriched metadata + group-aware split generation


import re
import json
import numpy as np
import pandas as pd
from pathlib import Path


# This cell assumes have already been run.
# It uses:
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - TRAIN_RATIO / VAL_RATIO / TEST_RATIO
# - SEED



# CHANGE THIS IF NEEDED
# Input / output files

SELECTED_METADATA_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun.csv"

ENRICHED_METADATA_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_enriched.csv"
SPLIT_METADATA_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_split.csv"

ENRICHED_SUMMARY_JSON = EXPERIMENT_RUN_DIR / "metadata_fullrun_enriched_summary.json"
SPLIT_SUMMARY_JSON = EXPERIMENT_RUN_DIR / "metadata_fullrun_split_summary.json"


# Load selected metadata

metadata_df = pd.read_csv(SELECTED_METADATA_CSV)

print("Loaded selected metadata:", metadata_df.shape)


# Regex patterns

CELEB_REAL_PATTERN = re.compile(r"^(id\d+)_(\d+)$")
CELEB_FAKE_PATTERN = re.compile(r"^(id\d+)_(id\d+)_(\d+)$")


# Identity/group extraction

def extract_identity_fields(row):
    dataset = row["dataset"]
    subset = row["source_subset"]
    stem = row["stem"]

    identity_primary = None
    identity_secondary = None
    identity_pair = None
    group_id = None
    clip_index = None
    parse_status = "unparsed"

    if dataset == "CelebDF":
        if subset in {"Celeb-real", "YouTube-real"}:
            m = CELEB_REAL_PATTERN.match(stem)
            if m:
                identity_primary = m.group(1)
                clip_index = m.group(2)
                identity_pair = identity_primary
                group_id = identity_primary
                parse_status = "parsed_celeb_real"
            else:
                group_id = stem
                parse_status = "fallback_celeb_real"

        elif subset == "Celeb-synthesis":
            m = CELEB_FAKE_PATTERN.match(stem)
            if m:
                identity_primary = m.group(1)
                identity_secondary = m.group(2)
                clip_index = m.group(3)
                identity_pair = f"{identity_primary}__{identity_secondary}"
                group_id = identity_pair
                parse_status = "parsed_celeb_fake"
            else:
                group_id = stem
                parse_status = "fallback_celeb_fake"

        else:
            group_id = stem
            parse_status = "unknown_celeb_subset"

    elif dataset == "FaceForensicsPP":
        group_id = stem
        parse_status = "ffpp_stem_group"

        pair_match = re.match(r"^(\d+)[_\-](\d+)$", stem)
        single_match = re.match(r"^(\d+)$", stem)

        if pair_match:
            identity_primary = pair_match.group(1)
            identity_secondary = pair_match.group(2)
            identity_pair = f"{identity_primary}__{identity_secondary}"
            group_id = identity_pair
            parse_status = "parsed_ffpp_pair"
        elif single_match:
            identity_primary = single_match.group(1)
            identity_pair = identity_primary
            group_id = identity_primary
            parse_status = "parsed_ffpp_single"

    else:
        group_id = stem
        parse_status = "unknown_dataset"

    return pd.Series({
        "identity_primary": identity_primary,
        "identity_secondary": identity_secondary,
        "identity_pair": identity_pair,
        "group_id": group_id,
        "clip_index": clip_index,
        "parse_status": parse_status,
    })

identity_cols = metadata_df.apply(extract_identity_fields, axis=1)
metadata_df = pd.concat([metadata_df, identity_cols], axis=1)


# Validation checks

print("\nDuplicate checks:")
print("  duplicate sample_id :", int(metadata_df["sample_id"].duplicated().sum()))
print("  duplicate file_path :", int(metadata_df["file_path"].duplicated().sum()))

print("\nParse-status distribution:")
print(metadata_df["parse_status"].value_counts(dropna=False))

print("\nUnique groups by dataset:")
print(metadata_df.groupby("dataset")["group_id"].nunique())

print("\nUnique primary identities by dataset:")
print(metadata_df.groupby("dataset")["identity_primary"].nunique(dropna=True))


# Save enriched metadata

metadata_df.to_csv(ENRICHED_METADATA_CSV, index=False)

enriched_summary = {
    "metadata_shape": [int(metadata_df.shape[0]), int(metadata_df.shape[1])],
    "parse_status_distribution": {
        str(k): int(v)
        for k, v in metadata_df["parse_status"].value_counts(dropna=False).to_dict().items()
    },
    "unique_groups_by_dataset": {
        str(k): int(v)
        for k, v in metadata_df.groupby("dataset")["group_id"].nunique().to_dict().items()
    },
    "unique_identity_primary_by_dataset": {
        str(k): int(v)
        for k, v in metadata_df.groupby("dataset")["identity_primary"].nunique(dropna=True).to_dict().items()
    },
}

with open(ENRICHED_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(enriched_summary, f, indent=2)


# Build group-level split table
# Group by (dataset, family_label_name, group_id)

group_df = (
    metadata_df.groupby(["dataset", "family_label_name", "group_id"], as_index=False)
    .agg(
        n_videos=("sample_id", "count"),
        binary_label_name=("binary_label_name", "first"),
        source_subset=("source_subset", lambda x: sorted(set(x))[0]),
    )
)

group_df = group_df.sort_values(
    by=["dataset", "family_label_name", "group_id"]
).reset_index(drop=True)

print("\nGroup-level table shape:", group_df.shape)


# Greedy reproducible split assignment within each
# (dataset, family_label_name) bucket

assignments = []

for (dataset_name, family_name), sub_df in group_df.groupby(["dataset", "family_label_name"], sort=True):
    sub_df = sub_df.copy().reset_index(drop=True)

    # shuffle groups reproducibly
    sub_df = sub_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    n_groups = len(sub_df)

    n_train = int(round(TRAIN_RATIO * n_groups))
    n_val = int(round(VAL_RATIO * n_groups))
    n_test = n_groups - n_train - n_val

    # safety correction
    if n_groups >= 3:
        if n_train == 0:
            n_train = 1
        if n_val == 0:
            n_val = 1
        if n_test == 0:
            n_test = 1

        while (n_train + n_val + n_test) > n_groups:
            if n_train >= max(n_val, n_test) and n_train > 1:
                n_train -= 1
            elif n_val >= n_test and n_val > 1:
                n_val -= 1
            elif n_test > 1:
                n_test -= 1
            else:
                break
    else:
        if n_groups == 1:
            n_train, n_val, n_test = 1, 0, 0
        elif n_groups == 2:
            n_train, n_val, n_test = 1, 0, 1

    split_labels = (
        ["train"] * n_train +
        ["val"] * n_val +
        ["test"] * n_test
    )

    split_labels = split_labels[:n_groups]
    if len(split_labels) < n_groups:
        split_labels += ["train"] * (n_groups - len(split_labels))

    sub_df["split"] = split_labels
    assignments.append(sub_df)

group_split_df = pd.concat(assignments, axis=0).reset_index(drop=True)

print("Assigned group-level split table:", group_split_df.shape)


# Merge split labels back to video-level metadata

split_df = metadata_df.merge(
    group_split_df[["dataset", "family_label_name", "group_id", "split"]],
    on=["dataset", "family_label_name", "group_id"],
    how="left",
    validate="many_to_one"
)

if split_df["split"].isna().any():
    missing_rows = split_df[split_df["split"].isna()]
    raise RuntimeError(f"Some rows did not receive split assignment: {len(missing_rows)}")

split_df = split_df.sort_values(
    by=["split", "dataset", "family_label_name", "group_id", "relative_path"]
).reset_index(drop=True)

split_df.to_csv(SPLIT_METADATA_CSV, index=False)


# Leakage check: no group overlap within same dataset/family

leakage_found = False
leakage_report = []

for (dataset_name, family_name), sub_df in split_df.groupby(["dataset", "family_label_name"]):
    train_groups = set(sub_df.loc[sub_df["split"] == "train", "group_id"])
    val_groups = set(sub_df.loc[sub_df["split"] == "val", "group_id"])
    test_groups = set(sub_df.loc[sub_df["split"] == "test", "group_id"])

    overlap_train_val = train_groups.intersection(val_groups)
    overlap_train_test = train_groups.intersection(test_groups)
    overlap_val_test = val_groups.intersection(test_groups)

    if overlap_train_val or overlap_train_test or overlap_val_test:
        leakage_found = True
        leakage_report.append({
            "dataset": dataset_name,
            "family": family_name,
            "train_val_overlap": len(overlap_train_val),
            "train_test_overlap": len(overlap_train_test),
            "val_test_overlap": len(overlap_val_test),
        })


# Save split summary

split_summary = {
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "n_video_rows": int(len(split_df)),
    "n_group_rows": int(len(group_split_df)),
    "video_split_counts": {
        str(k): int(v) for k, v in split_df["split"].value_counts().to_dict().items()
    },
    "group_split_counts": {
        str(k): int(v) for k, v in group_split_df["split"].value_counts().to_dict().items()
    },
    "video_split_by_dataset": {
        f"{k[0]}__{k[1]}": int(v)
        for k, v in split_df.groupby(["split", "dataset"]).size().to_dict().items()
    },
    "video_split_by_family": {
        f"{k[0]}__{k[1]}": int(v)
        for k, v in split_df.groupby(["split", "family_label_name"]).size().to_dict().items()
    },
    "leakage_found": bool(leakage_found),
    "leakage_report": leakage_report,
}

with open(SPLIT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(split_summary, f, indent=2)


# Print split summary

print("\nVideo-level split counts:")
print(split_df["split"].value_counts())

print("\nGroup-level split counts:")
print(group_split_df["split"].value_counts())

print("\nVideo-level split x dataset:")
print(pd.crosstab(split_df["split"], split_df["dataset"]))

print("\nVideo-level split x family:")
print(pd.crosstab(split_df["split"], split_df["family_label_name"]))

print("\nLeakage found?", leakage_found)
if leakage_report:
    print(leakage_report)

print("\nSaved files:")
print("  Enriched metadata CSV :", ENRICHED_METADATA_CSV)
print("  Split metadata CSV    :", SPLIT_METADATA_CSV)
print("  Enriched summary JSON :", ENRICHED_SUMMARY_JSON)
print("  Split summary JSON    :", SPLIT_SUMMARY_JSON)

print("\nPreview of split metadata:")
display(
    split_df[
        ["sample_id", "dataset", "source_subset", "family_label_name", "group_id", "split", "file_name"]
    ].head(20)
)

In [ ]:

# Full-run video probing


import os
import json
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path


# CHANGE THI IF NEEDED
# Input split metadata

FULL_SPLIT_METADATA_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_split.csv"


# Output files

FULL_PROBE_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_probed.csv"
FULL_PROBE_SUMMARY_JSON = EXPERIMENT_RUN_DIR / "metadata_fullrun_video_probe_summary.json"


# CHANGE THIS IF NEEDED
# Optional cap for debugging / staged full run
# Set to None to probe all full-run videos

PROBE_MAX_VIDEOS_FULL = None


# Load split metadata

split_df = pd.read_csv(FULL_SPLIT_METADATA_CSV)

if PROBE_MAX_VIDEOS_FULL is not None:
    split_df = split_df.head(PROBE_MAX_VIDEOS_FULL).copy()

print("Videos selected for probing:", len(split_df))


# Video probing helper

def probe_video(video_path):
    result = {
        "probe_ok": 0,
        "probe_error": None,
        "exists_on_disk": int(Path(video_path).exists()),
        "file_size_bytes": None,
        "fps": None,
        "frame_count": None,
        "width": None,
        "height": None,
        "duration_sec": None,
        "read_test_ok": 0,
    }

    video_path = str(video_path)

    if not Path(video_path).exists():
        result["probe_error"] = "file_not_found"
        return result

    try:
        result["file_size_bytes"] = int(Path(video_path).stat().st_size)
    except Exception as e:
        result["probe_error"] = f"stat_error: {e}"
        return result

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        result["probe_error"] = "opencv_open_failed"
        cap.release()
        return result

    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

        result["fps"] = float(fps) if fps is not None else None
        result["frame_count"] = int(frame_count) if frame_count is not None else None
        result["width"] = int(width) if width is not None else None
        result["height"] = int(height) if height is not None else None

        if fps is not None and fps > 0 and frame_count is not None and frame_count >= 0:
            result["duration_sec"] = float(frame_count / fps)
        else:
            result["duration_sec"] = None

        ok, frame = cap.read()
        result["read_test_ok"] = int(bool(ok and frame is not None))
        result["probe_ok"] = 1

    except Exception as e:
        result["probe_error"] = f"opencv_probe_error: {e}"

    finally:
        cap.release()

    return result


# Run probing

probe_rows = []
start_time = time.time()

for idx, row in split_df.iterrows():
    video_path = row["file_path"]
    probe_info = probe_video(video_path)

    merged = dict(row)
    merged.update(probe_info)
    probe_rows.append(merged)

    if (idx + 1) % 250 == 0 or (idx + 1) == len(split_df):
        print(f"Probed {idx + 1}/{len(split_df)} videos")

probe_df = pd.DataFrame(probe_rows)

elapsed = time.time() - start_time
print(f"\nProbing finished in {elapsed:.2f} seconds")


# Save probed metadata

probe_df.to_csv(FULL_PROBE_CSV, index=False)


# Summary diagnostics

n_total = len(probe_df)
n_probe_ok = int(probe_df["probe_ok"].sum())
n_read_ok = int(probe_df["read_test_ok"].sum())
n_missing = int((probe_df["exists_on_disk"] == 0).sum())
n_failed = int((probe_df["probe_ok"] == 0).sum())

summary = {
    "n_total_videos": int(n_total),
    "n_probe_ok": int(n_probe_ok),
    "n_read_test_ok": int(n_read_ok),
    "n_missing_on_disk": int(n_missing),
    "n_probe_failed": int(n_failed),
    "elapsed_seconds": float(elapsed),
}

numeric_cols = ["file_size_bytes", "fps", "frame_count", "width", "height", "duration_sec"]
for col in numeric_cols:
    valid = probe_df[col].dropna()
    if len(valid) > 0:
        summary[f"{col}_min"] = float(valid.min())
        summary[f"{col}_max"] = float(valid.max())
        summary[f"{col}_mean"] = float(valid.mean())

with open(FULL_PROBE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nProbe summary:")
print("  Total videos        :", n_total)
print("  Probe OK            :", n_probe_ok)
print("  Read test OK        :", n_read_ok)
print("  Missing on disk     :", n_missing)
print("  Probe failed        :", n_failed)

print("\nDataset x split counts:")
print(probe_df.groupby(["split", "dataset"]).size())

print("\nFamily x split counts:")
print(probe_df.groupby(["split", "family_label_name"]).size())

print("\nProbe failures preview:")
display(
    probe_df.loc[probe_df["probe_ok"] == 0, [
        "dataset", "source_subset", "split", "file_name", "file_path", "probe_error"
    ]].head(20)
)

print("\nBasic numeric summary:")
display(
    probe_df[["file_size_bytes", "fps", "frame_count", "width", "height", "duration_sec"]].describe()
)

print("\nSaved:")
print("  Probed metadata CSV   :", FULL_PROBE_CSV)
print("  Probe summary JSON    :", FULL_PROBE_SUMMARY_JSON)

In [ ]:

# Full-run uniform frame extraction


import os
import json
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path


# This cell assumes already defined:
# - PROCESSED_ROOT
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - FRAMES_PER_VIDEO
# - OUTPUT_IMAGE_EXT
# - JPEG_QUALITY



# CHANGE THIS IF NEEDED
# Input probed metadata

FULL_PROBED_METADATA_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_probed.csv"


# CHANGE THIS IF NEEDED
# Output locations

FRAMES_ROOT_FULL = PROCESSED_ROOT / "fullrun_frames_uniform"
FRAME_MANIFEST_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_frames_uniform.csv"
FRAME_EXTRACTION_SUMMARY_JSON_FULL = EXPERIMENT_RUN_DIR / "frame_extraction_fullrun_summary.json"
FRAME_EXTRACTION_FAILURES_JSON_FULL = EXPERIMENT_RUN_DIR / "frame_extraction_fullrun_failures.json"


# CHANGE THIS IF NEEDED
# Extraction controls

OVERWRITE_EXISTING = False

# Optional cap for staged execution
# Set to None to process all videos
EXTRACT_MAX_VIDEOS_FULL = None


# Load probed metadata

probe_df = pd.read_csv(FULL_PROBED_METADATA_CSV)

# Keep only valid readable videos
probe_df = probe_df[(probe_df["probe_ok"] == 1) & (probe_df["read_test_ok"] == 1)].copy()

if EXTRACT_MAX_VIDEOS_FULL is not None:
    probe_df = probe_df.head(EXTRACT_MAX_VIDEOS_FULL).copy()

print("Videos selected for frame extraction:", len(probe_df))

FRAMES_ROOT_FULL.mkdir(parents=True, exist_ok=True)


# Helpers

def safe_str(x):
    return str(x).replace("\\", "_").replace("/", "_").replace(":", "_").replace(" ", "_")

def compute_uniform_indices(frame_count, n_samples):
    if frame_count is None or frame_count <= 0:
        return []

    if frame_count < n_samples:
        idxs = np.linspace(0, max(frame_count - 1, 0), num=n_samples)
        idxs = np.clip(np.round(idxs).astype(int), 0, max(frame_count - 1, 0))
        return idxs.tolist()

    idxs = np.linspace(0.05, 0.95, num=n_samples) * (frame_count - 1)
    idxs = np.clip(np.round(idxs).astype(int), 0, frame_count - 1)
    return idxs.tolist()

def read_frame_at_index(cap, frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame_bgr = cap.read()
    return ok, frame_bgr


# Extraction loop

frame_rows = []
start_time = time.time()

n_videos_ok = 0
n_videos_failed = 0
video_failures = []

for vid_idx, row in probe_df.iterrows():
    video_path = row["file_path"]
    sample_id = row["sample_id"]
    dataset = row["dataset"]
    split = row["split"]
    family = row["family_label_name"]
    frame_count = int(row["frame_count"]) if pd.notna(row["frame_count"]) else None

    sampled_indices = compute_uniform_indices(frame_count, FRAMES_PER_VIDEO)

    out_dir = FRAMES_ROOT_FULL / safe_str(split) / safe_str(dataset) / safe_str(family) / safe_str(sample_id)
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        n_videos_failed += 1
        video_failures.append({
            "sample_id": sample_id,
            "file_path": video_path,
            "error": "opencv_open_failed_during_extraction",
        })
        continue

    video_saved_any = False

    try:
        for local_frame_idx, source_frame_idx in enumerate(sampled_indices):
            out_name = f"frame_{local_frame_idx:02d}_src_{int(source_frame_idx):05d}{OUTPUT_IMAGE_EXT}"
            out_path = out_dir / out_name

            if out_path.exists() and not OVERWRITE_EXISTING:
                frame_rows.append({
                    **row.to_dict(),
                    "frame_index_local": int(local_frame_idx),
                    "frame_index_source": int(source_frame_idx),
                    "frame_path": str(out_path.resolve()),
                    "frame_file_name": out_name,
                    "frame_ext": OUTPUT_IMAGE_EXT,
                    "frame_saved": 1,
                    "frame_read_ok": 1,
                    "frame_height": None,
                    "frame_width": None,
                    "extraction_error": None,
                })
                video_saved_any = True
                continue

            ok, frame_bgr = read_frame_at_index(cap, source_frame_idx)

            if not ok or frame_bgr is None:
                frame_rows.append({
                    **row.to_dict(),
                    "frame_index_local": int(local_frame_idx),
                    "frame_index_source": int(source_frame_idx),
                    "frame_path": str(out_path.resolve()),
                    "frame_file_name": out_name,
                    "frame_ext": OUTPUT_IMAGE_EXT,
                    "frame_saved": 0,
                    "frame_read_ok": 0,
                    "frame_height": None,
                    "frame_width": None,
                    "extraction_error": "frame_read_failed",
                })
                continue

            h, w = frame_bgr.shape[:2]

            if OUTPUT_IMAGE_EXT.lower() in [".jpg", ".jpeg"]:
                ok_write = cv2.imwrite(
                    str(out_path),
                    frame_bgr,
                    [int(cv2.IMWRITE_JPEG_QUALITY), int(JPEG_QUALITY)]
                )
            elif OUTPUT_IMAGE_EXT.lower() == ".png":
                ok_write = cv2.imwrite(str(out_path), frame_bgr)
            else:
                raise ValueError(f"Unsupported OUTPUT_IMAGE_EXT: {OUTPUT_IMAGE_EXT}")

            frame_rows.append({
                **row.to_dict(),
                "frame_index_local": int(local_frame_idx),
                "frame_index_source": int(source_frame_idx),
                "frame_path": str(out_path.resolve()),
                "frame_file_name": out_name,
                "frame_ext": OUTPUT_IMAGE_EXT,
                "frame_saved": int(bool(ok_write)),
                "frame_read_ok": 1,
                "frame_height": int(h),
                "frame_width": int(w),
                "extraction_error": None if ok_write else "frame_write_failed",
            })

            if ok_write:
                video_saved_any = True

        if video_saved_any:
            n_videos_ok += 1
        else:
            n_videos_failed += 1
            video_failures.append({
                "sample_id": sample_id,
                "file_path": video_path,
                "error": "no_frames_saved",
            })

    except Exception as e:
        n_videos_failed += 1
        video_failures.append({
            "sample_id": sample_id,
            "file_path": video_path,
            "error": f"exception: {e}",
        })

    finally:
        cap.release()

    if (vid_idx + 1) % 250 == 0 or (vid_idx + 1) == len(probe_df):
        print(f"Processed {vid_idx + 1}/{len(probe_df)} videos")

elapsed = time.time() - start_time

frame_df = pd.DataFrame(frame_rows)
frame_df.to_csv(FRAME_MANIFEST_CSV_FULL, index=False)

with open(FRAME_EXTRACTION_FAILURES_JSON_FULL, "w", encoding="utf-8") as f:
    json.dump(video_failures, f, indent=2)


# Summary

summary = {
    "n_input_videos": int(len(probe_df)),
    "frames_per_video_target": int(FRAMES_PER_VIDEO),
    "n_videos_ok": int(n_videos_ok),
    "n_videos_failed": int(n_videos_failed),
    "n_total_frame_rows": int(len(frame_df)),
    "n_frames_saved": int(frame_df["frame_saved"].sum()) if len(frame_df) > 0 else 0,
    "n_frame_read_ok": int(frame_df["frame_read_ok"].sum()) if len(frame_df) > 0 else 0,
    "output_root": str(FRAMES_ROOT_FULL),
    "frame_manifest_csv": str(FRAME_MANIFEST_CSV_FULL),
    "elapsed_seconds": float(elapsed),
}

with open(FRAME_EXTRACTION_SUMMARY_JSON_FULL, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nFrame extraction summary:")
print("  Input videos          :", len(probe_df))
print("  Videos OK             :", n_videos_ok)
print("  Videos failed         :", n_videos_failed)
print("  Frame rows            :", len(frame_df))
print("  Frames saved          :", int(frame_df['frame_saved'].sum()) if len(frame_df) > 0 else 0)
print("  Frame read OK         :", int(frame_df['frame_read_ok'].sum()) if len(frame_df) > 0 else 0)
print("  Elapsed seconds       :", round(elapsed, 2))

print("\nFrames per split:")
print(frame_df.groupby("split")["frame_saved"].sum())

print("\nFrames per dataset:")
print(frame_df.groupby("dataset")["frame_saved"].sum())

print("\nFrame extraction errors preview:")
display(
    frame_df.loc[frame_df["frame_saved"] == 0, [
        "dataset", "split", "file_name", "frame_file_name", "frame_index_source", "extraction_error"
    ]].head(20)
)

print("\nSaved:")
print("  Frames root            :", FRAMES_ROOT_FULL)
print("  Frame manifest CSV     :", FRAME_MANIFEST_CSV_FULL)
print("  Failure log JSON       :", FRAME_EXTRACTION_FAILURES_JSON_FULL)
print("  Summary JSON           :", FRAME_EXTRACTION_SUMMARY_JSON_FULL)

In [ ]:

# Full-run face extraction with MTCNN


import os
import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
from facenet_pytorch import MTCNN


# This cell assumes already defined:
# - PROCESSED_ROOT
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - FACE_IMAGE_SIZE
# - FACE_CONF_THRESHOLD
# - FACE_MARGIN_RATIO
# - KEEP_ONLY_LARGEST_FACE



# CHANGE THIS IF NEEDED
# Input frame manifest

FRAME_MANIFEST_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_frames_uniform.csv"


# CHANGE THIS IF NEEDED
# Output locations

FACES_ROOT_FULL = PROCESSED_ROOT / "fullrun_faces_mtcnn"
FACE_MANIFEST_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_faces_mtcnn.csv"
FACE_SUMMARY_JSON_FULL = EXPERIMENT_RUN_DIR / "face_extraction_fullrun_summary.json"
FACE_FAILURES_JSON_FULL = EXPERIMENT_RUN_DIR / "face_extraction_fullrun_failures.json"


# CHANGE THIS IF NEEDED
# Controls

OVERWRITE_EXISTING_FACES = False

# Optional cap for staged execution
# Set to None to process all extracted frames
MAX_FRAMES_TO_PROCESS_FULL = None


# Device

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# Load frame manifest

frame_df = pd.read_csv(FRAME_MANIFEST_CSV_FULL)
frame_df = frame_df[frame_df["frame_saved"] == 1].copy()

if MAX_FRAMES_TO_PROCESS_FULL is not None:
    frame_df = frame_df.head(MAX_FRAMES_TO_PROCESS_FULL).copy()

print("Frames selected for face extraction:", len(frame_df))

FACES_ROOT_FULL.mkdir(parents=True, exist_ok=True)


# Initialize detector

mtcnn = MTCNN(
    image_size=FACE_IMAGE_SIZE,
    margin=0,
    keep_all=True,
    post_process=False,
    device=DEVICE,
)


# Helpers

def safe_str(x):
    return str(x).replace("\\", "_").replace("/", "_").replace(":", "_").replace(" ", "_")

def expand_box(box, img_w, img_h, margin_ratio=0.20):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1

    mx = bw * margin_ratio
    my = bh * margin_ratio

    nx1 = max(0, int(round(x1 - mx)))
    ny1 = max(0, int(round(y1 - my)))
    nx2 = min(img_w - 1, int(round(x2 + mx)))
    ny2 = min(img_h - 1, int(round(y2 + my)))

    return nx1, ny1, nx2, ny2

def choose_face(boxes, probs):
    if boxes is None or len(boxes) == 0:
        return None, None, None

    boxes = np.asarray(boxes)
    probs = np.asarray(probs) if probs is not None else np.array([None] * len(boxes), dtype=object)

    valid_indices = []
    for i in range(len(boxes)):
        p = probs[i]
        if p is None or (isinstance(p, float) and np.isnan(p)):
            continue
        if float(p) >= FACE_CONF_THRESHOLD:
            valid_indices.append(i)

    if len(valid_indices) == 0:
        return None, None, None

    if KEEP_ONLY_LARGEST_FACE:
        areas = []
        for i in valid_indices:
            x1, y1, x2, y2 = boxes[i]
            areas.append(max(0.0, (x2 - x1)) * max(0.0, (y2 - y1)))
        best_idx = valid_indices[int(np.argmax(areas))]
    else:
        best_idx = valid_indices[0]

    return best_idx, boxes[best_idx], float(probs[best_idx])


# Extraction loop

face_rows = []
failures = []
start_time = time.time()

n_face_ok = 0
n_face_fail = 0

for idx, row in frame_df.iterrows():
    frame_path = Path(row["frame_path"])
    sample_id = row["sample_id"]
    split = row["split"]
    dataset = row["dataset"]
    family = row["family_label_name"]
    frame_index_local = int(row["frame_index_local"])
    frame_index_source = int(row["frame_index_source"])

    out_dir = FACES_ROOT_FULL / safe_str(split) / safe_str(dataset) / safe_str(family) / safe_str(sample_id)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_name = f"face_{frame_index_local:02d}_src_{frame_index_source:05d}.jpg"
    out_path = out_dir / out_name

    if out_path.exists() and not OVERWRITE_EXISTING_FACES:
        face_rows.append({
            **row.to_dict(),
            "face_detected": 1,
            "face_saved": 1,
            "face_confidence": None,
            "face_bbox_x1": None,
            "face_bbox_y1": None,
            "face_bbox_x2": None,
            "face_bbox_y2": None,
            "face_crop_path": str(out_path.resolve()),
            "face_crop_file_name": out_name,
            "face_crop_size": FACE_IMAGE_SIZE,
            "face_detection_error": None,
        })
        n_face_ok += 1
        continue

    try:
        if not frame_path.exists():
            face_rows.append({
                **row.to_dict(),
                "face_detected": 0,
                "face_saved": 0,
                "face_confidence": None,
                "face_bbox_x1": None,
                "face_bbox_y1": None,
                "face_bbox_x2": None,
                "face_bbox_y2": None,
                "face_crop_path": str(out_path.resolve()),
                "face_crop_file_name": out_name,
                "face_crop_size": FACE_IMAGE_SIZE,
                "face_detection_error": "frame_not_found",
            })
            n_face_fail += 1
            continue

        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            face_rows.append({
                **row.to_dict(),
                "face_detected": 0,
                "face_saved": 0,
                "face_confidence": None,
                "face_bbox_x1": None,
                "face_bbox_y1": None,
                "face_bbox_x2": None,
                "face_bbox_y2": None,
                "face_crop_path": str(out_path.resolve()),
                "face_crop_file_name": out_name,
                "face_crop_size": FACE_IMAGE_SIZE,
                "face_detection_error": "frame_read_failed",
            })
            n_face_fail += 1
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_h, img_w = img_rgb.shape[:2]

        pil_img = Image.fromarray(img_rgb)
        boxes, probs = mtcnn.detect(pil_img)

        best_idx, best_box, best_prob = choose_face(boxes, probs)

        if best_box is None:
            face_rows.append({
                **row.to_dict(),
                "face_detected": 0,
                "face_saved": 0,
                "face_confidence": None,
                "face_bbox_x1": None,
                "face_bbox_y1": None,
                "face_bbox_x2": None,
                "face_bbox_y2": None,
                "face_crop_path": str(out_path.resolve()),
                "face_crop_file_name": out_name,
                "face_crop_size": FACE_IMAGE_SIZE,
                "face_detection_error": "no_face_above_threshold",
            })
            n_face_fail += 1
            continue

        x1, y1, x2, y2 = expand_box(best_box, img_w, img_h, margin_ratio=FACE_MARGIN_RATIO)
        face_crop = img_rgb[y1:y2, x1:x2]

        if face_crop is None or face_crop.size == 0:
            face_rows.append({
                **row.to_dict(),
                "face_detected": 0,
                "face_saved": 0,
                "face_confidence": float(best_prob),
                "face_bbox_x1": int(x1),
                "face_bbox_y1": int(y1),
                "face_bbox_x2": int(x2),
                "face_bbox_y2": int(y2),
                "face_crop_path": str(out_path.resolve()),
                "face_crop_file_name": out_name,
                "face_crop_size": FACE_IMAGE_SIZE,
                "face_detection_error": "empty_face_crop",
            })
            n_face_fail += 1
            continue

        face_crop_resized = cv2.resize(face_crop, (FACE_IMAGE_SIZE, FACE_IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        face_crop_bgr = cv2.cvtColor(face_crop_resized, cv2.COLOR_RGB2BGR)

        ok_write = cv2.imwrite(str(out_path), face_crop_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

        face_rows.append({
            **row.to_dict(),
            "face_detected": 1,
            "face_saved": int(bool(ok_write)),
            "face_confidence": float(best_prob),
            "face_bbox_x1": int(x1),
            "face_bbox_y1": int(y1),
            "face_bbox_x2": int(x2),
            "face_bbox_y2": int(y2),
            "face_crop_path": str(out_path.resolve()),
            "face_crop_file_name": out_name,
            "face_crop_size": FACE_IMAGE_SIZE,
            "face_detection_error": None if ok_write else "face_write_failed",
        })

        if ok_write:
            n_face_ok += 1
        else:
            n_face_fail += 1

    except Exception as e:
        face_rows.append({
            **row.to_dict(),
            "face_detected": 0,
            "face_saved": 0,
            "face_confidence": None,
            "face_bbox_x1": None,
            "face_bbox_y1": None,
            "face_bbox_x2": None,
            "face_bbox_y2": None,
            "face_crop_path": str(out_path.resolve()),
            "face_crop_file_name": out_name,
            "face_crop_size": FACE_IMAGE_SIZE,
            "face_detection_error": f"exception: {e}",
        })
        failures.append({
            "sample_id": sample_id,
            "frame_path": str(frame_path),
            "error": str(e),
        })
        n_face_fail += 1

    if (idx + 1) % 500 == 0 or (idx + 1) == len(frame_df):
        print(f"Processed {idx + 1}/{len(frame_df)} frames")

elapsed = time.time() - start_time

face_df = pd.DataFrame(face_rows)
face_df.to_csv(FACE_MANIFEST_CSV_FULL, index=False)

with open(FACE_FAILURES_JSON_FULL, "w", encoding="utf-8") as f:
    json.dump(failures, f, indent=2)


# Summary

summary = {
    "n_input_frames": int(len(frame_df)),
    "n_face_detected": int(face_df["face_detected"].sum()) if len(face_df) > 0 else 0,
    "n_face_saved": int(face_df["face_saved"].sum()) if len(face_df) > 0 else 0,
    "n_face_failed": int((face_df["face_saved"] == 0).sum()) if len(face_df) > 0 else 0,
    "face_conf_threshold": float(FACE_CONF_THRESHOLD),
    "face_margin_ratio": float(FACE_MARGIN_RATIO),
    "face_image_size": int(FACE_IMAGE_SIZE),
    "device": DEVICE,
    "elapsed_seconds": float(elapsed),
}

with open(FACE_SUMMARY_JSON_FULL, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nFace extraction summary:")
print("  Input frames          :", len(frame_df))
print("  Faces detected        :", int(face_df["face_detected"].sum()) if len(face_df) > 0 else 0)
print("  Faces saved           :", int(face_df["face_saved"].sum()) if len(face_df) > 0 else 0)
print("  Face failures         :", int((face_df["face_saved"] == 0).sum()) if len(face_df) > 0 else 0)
print("  Elapsed seconds       :", round(elapsed, 2))

print("\nFaces saved per split:")
print(face_df.groupby("split")["face_saved"].sum())

print("\nFaces saved per dataset:")
print(face_df.groupby("dataset")["face_saved"].sum())

print("\nDetection error preview:")
display(
    face_df.loc[face_df["face_saved"] == 0, [
        "dataset", "split", "file_name", "frame_file_name", "face_detection_error"
    ]].head(20)
)

print("\nSaved:")
print("  Faces root            :", FACES_ROOT_FULL)
print("  Face manifest CSV     :", FACE_MANIFEST_CSV_FULL)
print("  Face failures JSON    :", FACE_FAILURES_JSON_FULL)
print("  Face summary JSON     :", FACE_SUMMARY_JSON_FULL)

In [ ]:

# Full-run model-ready completeness filtering


import os
import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd


# This cell assumes already defined:
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - FRAMES_PER_VIDEO



# CHANGE THIS IF NEEDED
# Input face manifest

FACE_MANIFEST_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_faces_mtcnn.csv"


# Output files

MODEL_READY_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"
INCOMPLETE_SAMPLES_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_incomplete_samples.csv"
MODEL_READY_SUMMARY_JSON_FULL = EXPERIMENT_RUN_DIR / "model_ready_fullrun_summary.json"


# CHANGE THIS IF NEEDED
# Expected number of face crops per video

EXPECTED_FACES_PER_VIDEO = FRAMES_PER_VIDEO


# Optional cap for staged debugging
# Set to None to use all rows

MAX_FACE_ROWS_TO_CHECK_FULL = None


# Load face manifest

face_df = pd.read_csv(FACE_MANIFEST_CSV_FULL, low_memory=False)

if MAX_FACE_ROWS_TO_CHECK_FULL is not None:
    face_df = face_df.head(MAX_FACE_ROWS_TO_CHECK_FULL).copy()

print("Loaded face manifest:", face_df.shape)

# Keep only rows where a face crop was saved
face_ok_df = face_df[face_df["face_saved"] == 1].copy()
print("Rows with saved face crops:", face_ok_df.shape)


# Per-face quality probing

def probe_face_image(face_path):
    result = {
        "face_exists_on_disk": 0,
        "face_img_read_ok": 0,
        "face_img_h": None,
        "face_img_w": None,
        "face_img_channels": None,
        "face_img_mean": None,
        "face_img_std": None,
        "face_img_min": None,
        "face_img_max": None,
        "face_is_too_dark": 0,
        "face_is_too_bright": 0,
        "face_is_low_contrast": 0,
        "face_probe_error": None,
    }

    p = Path(face_path)
    if not p.exists():
        result["face_probe_error"] = "face_crop_missing"
        return result

    result["face_exists_on_disk"] = 1

    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        result["face_probe_error"] = "cv2_imread_failed"
        return result

    result["face_img_read_ok"] = 1

    h, w = img.shape[:2]
    c = img.shape[2] if len(img.shape) == 3 else 1

    img_float = img.astype(np.float32)

    mean_val = float(img_float.mean())
    std_val = float(img_float.std())
    min_val = float(img_float.min())
    max_val = float(img_float.max())

    result["face_img_h"] = int(h)
    result["face_img_w"] = int(w)
    result["face_img_channels"] = int(c)
    result["face_img_mean"] = mean_val
    result["face_img_std"] = std_val
    result["face_img_min"] = min_val
    result["face_img_max"] = max_val

    result["face_is_too_dark"] = int(mean_val < 35.0)
    result["face_is_too_bright"] = int(mean_val > 220.0)
    result["face_is_low_contrast"] = int(std_val < 20.0)

    return result


# Run QC probing

qc_rows = []
start_time = time.time()

for idx, row in face_ok_df.iterrows():
    qc = probe_face_image(row["face_crop_path"])
    qc_rows.append({**row.to_dict(), **qc})

    if (idx + 1) % 1000 == 0 or (idx + 1) == len(face_ok_df):
        print(f"QC checked {idx + 1}/{len(face_ok_df)} face crops")

qc_df = pd.DataFrame(qc_rows)

elapsed = time.time() - start_time
print(f"\nQC probing finished in {elapsed:.2f} seconds")


# Per-sample completeness check

sample_summary_df = (
    qc_df.groupby(
        ["sample_id", "dataset", "split", "family_label_name", "binary_label_name", "file_name"],
        as_index=False
    )
    .agg(
        n_face_rows=("face_crop_path", "count"),
        n_face_exists=("face_exists_on_disk", "sum"),
        n_face_img_read_ok=("face_img_read_ok", "sum"),
        mean_face_confidence=("face_confidence", "mean"),
        mean_face_img_mean=("face_img_mean", "mean"),
        mean_face_img_std=("face_img_std", "mean"),
        any_too_dark=("face_is_too_dark", "max"),
        any_too_bright=("face_is_too_bright", "max"),
        any_low_contrast=("face_is_low_contrast", "max"),
    )
)

sample_summary_df["is_complete_sample"] = (
    (sample_summary_df["n_face_rows"] == EXPECTED_FACES_PER_VIDEO) &
    (sample_summary_df["n_face_exists"] == EXPECTED_FACES_PER_VIDEO) &
    (sample_summary_df["n_face_img_read_ok"] == EXPECTED_FACES_PER_VIDEO)
).astype(int)

incomplete_df = sample_summary_df[sample_summary_df["is_complete_sample"] == 0].copy()


# Build final model-ready manifest

complete_sample_ids = set(sample_summary_df.loc[sample_summary_df["is_complete_sample"] == 1, "sample_id"])

model_ready_df = qc_df[qc_df["sample_id"].isin(complete_sample_ids)].copy()

model_ready_df = model_ready_df.sort_values(
    by=["split", "dataset", "family_label_name", "sample_id", "frame_index_local"]
).reset_index(drop=True)


# Save files

model_ready_df.to_csv(MODEL_READY_CSV_FULL, index=False)
incomplete_df.to_csv(INCOMPLETE_SAMPLES_CSV_FULL, index=False)


# Summary

summary = {
    "n_face_rows_input": int(len(face_df)),
    "n_face_rows_saved": int(len(face_ok_df)),
    "n_qc_rows": int(len(qc_df)),
    "n_unique_samples_total": int(sample_summary_df["sample_id"].nunique()),
    "n_complete_samples": int(sample_summary_df["is_complete_sample"].sum()),
    "n_incomplete_samples": int((sample_summary_df["is_complete_sample"] == 0).sum()),
    "expected_faces_per_video": int(EXPECTED_FACES_PER_VIDEO),
    "n_model_ready_rows": int(len(model_ready_df)),
    "elapsed_seconds": float(elapsed),
    "too_dark_count": int(qc_df["face_is_too_dark"].sum()) if len(qc_df) > 0 else 0,
    "too_bright_count": int(qc_df["face_is_too_bright"].sum()) if len(qc_df) > 0 else 0,
    "low_contrast_count": int(qc_df["face_is_low_contrast"].sum()) if len(qc_df) > 0 else 0,
}

with open(MODEL_READY_SUMMARY_JSON_FULL, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nModel-ready summary:")
print("  Unique samples total      :", sample_summary_df["sample_id"].nunique())
print("  Complete samples          :", int(sample_summary_df["is_complete_sample"].sum()))
print("  Incomplete samples        :", int((sample_summary_df["is_complete_sample"] == 0).sum()))
print("  Model-ready rows          :", len(model_ready_df))

print("\nComplete samples by split:")
print(sample_summary_df.groupby("split")["is_complete_sample"].sum())

print("\nComplete samples by dataset:")
print(sample_summary_df.groupby("dataset")["is_complete_sample"].sum())

print("\nComplete samples by family:")
print(sample_summary_df.groupby("family_label_name")["is_complete_sample"].sum())

print("\nQC flag counts:")
print("  Too dark          :", int(qc_df["face_is_too_dark"].sum()))
print("  Too bright        :", int(qc_df["face_is_too_bright"].sum()))
print("  Low contrast      :", int(qc_df["face_is_low_contrast"].sum()))

print("\nIncomplete samples preview:")
display(incomplete_df.head(20))

print("\nSaved:")
print("  Model-ready CSV         :", MODEL_READY_CSV_FULL)
print("  Incomplete samples CSV  :", INCOMPLETE_SAMPLES_CSV_FULL)
print("  Summary JSON            :", MODEL_READY_SUMMARY_JSON_FULL)

In [ ]:

# Full-run binary dataset and dataloaders


import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# This cell assumes already defined:
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - FACE_IMAGE_SIZE



# CHANGE THIS IF NEEDED
# Input model-ready manifest

MODEL_READY_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"


# CHANGE THIS IF NEEDED
# Data loading settings

FRAMES_PER_SAMPLE = 8
IMAGE_SIZE = FACE_IMAGE_SIZE
BATCH_SIZE = 8
NUM_WORKERS = 4 if os.name != "nt" else 0
PIN_MEMORY = torch.cuda.is_available()

# Optional subset for quick full-run debug
# Set to None to use all full-run samples
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None


# Load manifest

model_df = pd.read_csv(MODEL_READY_CSV_FULL, low_memory=False)

print("Loaded full model-ready manifest:", model_df.shape)

required_cols = [
    "sample_id", "split", "dataset", "family_label_name",
    "binary_label", "binary_label_name", "frame_index_local", "face_crop_path"
]
missing_cols = [c for c in required_cols if c not in model_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")


# Group rows into one video sample = 8 face crops

group_keys = [
    "sample_id", "split", "dataset", "family_label_name",
    "binary_label", "binary_label_name"
]

sample_groups = []

for key_vals, sub_df in model_df.groupby(group_keys):
    sub_df = sub_df.sort_values("frame_index_local").copy()

    frame_paths = sub_df["face_crop_path"].tolist()
    frame_indices = sub_df["frame_index_local"].tolist()

    if len(frame_paths) != FRAMES_PER_SAMPLE:
        continue

    if frame_indices != list(range(FRAMES_PER_SAMPLE)):
        continue

    sample_groups.append({
        "sample_id": key_vals[0],
        "split": key_vals[1],
        "dataset": key_vals[2],
        "family_label_name": key_vals[3],
        "binary_label": int(key_vals[4]),
        "binary_label_name": key_vals[5],
        "frame_paths": frame_paths,
    })

samples_df = pd.DataFrame(sample_groups)

print("Grouped video-level samples:", samples_df.shape)


# Split tables

train_samples_full = samples_df[samples_df["split"] == "train"].reset_index(drop=True)
val_samples_full = samples_df[samples_df["split"] == "val"].reset_index(drop=True)
test_samples_full = samples_df[samples_df["split"] == "test"].reset_index(drop=True)

if MAX_TRAIN_SAMPLES is not None:
    train_samples_full = train_samples_full.head(MAX_TRAIN_SAMPLES).copy()
if MAX_VAL_SAMPLES is not None:
    val_samples_full = val_samples_full.head(MAX_VAL_SAMPLES).copy()
if MAX_TEST_SAMPLES is not None:
    test_samples_full = test_samples_full.head(MAX_TEST_SAMPLES).copy()

print("\nVideo-level sample counts:")
print("  train:", len(train_samples_full))
print("  val  :", len(val_samples_full))
print("  test :", len(test_samples_full))

print("\nBinary distribution by split:")
print(samples_df.groupby(["split", "binary_label_name"]).size())

print("\nFamily distribution by split:")
print(samples_df.groupby(["split", "family_label_name"]).size())

print("\nDataset distribution by split:")
print(samples_df.groupby(["split", "dataset"]).size())


# Transforms

train_transform_full = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform_full = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# Dataset

class FullBinaryVideoFaceDataset(Dataset):
    def __init__(self, sample_table: pd.DataFrame, transform=None):
        self.sample_table = sample_table.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.sample_table)

    def __getitem__(self, idx):
        row = self.sample_table.iloc[idx]

        frame_tensors = []
        for frame_path in row["frame_paths"]:
            img = Image.open(frame_path).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
            frame_tensors.append(img)

        x = torch.stack(frame_tensors, dim=0)  # [T, C, H, W]
        y = torch.tensor(int(row["binary_label"]), dtype=torch.long)

        meta = {
            "sample_id": row["sample_id"],
            "split": row["split"],
            "dataset": row["dataset"],
            "family_label_name": row["family_label_name"],
            "binary_label_name": row["binary_label_name"],
        }

        return x, y, meta


# Datasets

train_dataset_full = FullBinaryVideoFaceDataset(train_samples_full, transform=train_transform_full)
val_dataset_full = FullBinaryVideoFaceDataset(val_samples_full, transform=eval_transform_full)
test_dataset_full = FullBinaryVideoFaceDataset(test_samples_full, transform=eval_transform_full)


# Custom collate

def custom_collate_full(batch):
    xs, ys, metas = zip(*batch)
    xs = torch.stack(xs, dim=0)   # [B, T, C, H, W]
    ys = torch.stack(ys, dim=0)   # [B]
    metas = list(metas)
    return xs, ys, metas


# Dataloaders

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=custom_collate_full,
)

val_loader_full = DataLoader(
    val_dataset_full,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=custom_collate_full,
)

test_loader_full = DataLoader(
    test_dataset_full,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=custom_collate_full,
)


# Sanity check batch

xb, yb, metab = next(iter(train_loader_full))

print("\nSanity check batch:")
print("  x shape:", tuple(xb.shape))
print("  y shape:", tuple(yb.shape))
print("  y dtype:", yb.dtype)
print("  batch label counts:", Counter(yb.cpu().numpy().tolist()))
print("  first meta:", metab[0])


# Save loader summary

loader_summary = {
    "n_train_samples": int(len(train_dataset_full)),
    "n_val_samples": int(len(val_dataset_full)),
    "n_test_samples": int(len(test_dataset_full)),
    "frames_per_sample": int(FRAMES_PER_SAMPLE),
    "image_size": int(IMAGE_SIZE),
    "batch_size": int(BATCH_SIZE),
    "num_workers": int(NUM_WORKERS),
    "pin_memory": bool(PIN_MEMORY),
    "train_binary_distribution": {
        str(k): int(v) for k, v in train_samples_full["binary_label_name"].value_counts().to_dict().items()
    },
    "val_binary_distribution": {
        str(k): int(v) for k, v in val_samples_full["binary_label_name"].value_counts().to_dict().items()
    },
    "test_binary_distribution": {
        str(k): int(v) for k, v in test_samples_full["binary_label_name"].value_counts().to_dict().items()
    },
}

loader_summary_json = EXPERIMENT_RUN_DIR / "binary_fullrun_dataset_loader_summary.json"
with open(loader_summary_json, "w", encoding="utf-8") as f:
    json.dump(loader_summary, f, indent=2)

print("\nSaved:")
print("  Loader summary JSON :", loader_summary_json)

In [ ]:

# Full-run binary baseline training


import os
import json
import copy
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


#  training configuration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 8
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
USE_CLASS_WEIGHT = True
MAX_GRAD_NORM = 1.0
FREEZE_BACKBONE = False


# Output paths

CHECKPOINT_DIR = CHECKPOINT_ROOT / "binary_baseline_resnet18_fullrun"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pt"
LAST_MODEL_PATH = CHECKPOINT_DIR / "last_model.pt"
TRAINING_HISTORY_JSON = CHECKPOINT_DIR / "training_history.json"

print("Using device:", DEVICE)


# Compute class weights

train_label_counts = train_samples_full["binary_label"].value_counts().sort_index().to_dict()
print("Train label counts:", train_label_counts)

if USE_CLASS_WEIGHT:
    n_neg = train_label_counts.get(0, 0)
    n_pos = train_label_counts.get(1, 0)
    total = n_neg + n_pos

    class_weights = torch.tensor([
        total / max(n_neg, 1),
        total / max(n_pos, 1)
    ], dtype=torch.float32, device=DEVICE)

    print("Using class weights:", class_weights.detach().cpu().numpy())
else:
    class_weights = None
    print("Class weighting disabled")


# Model

class ResNet18VideoBinaryClassifier(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=False):
        super().__init__()

        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, 2)
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        video_feats = feats.mean(dim=1)
        logits = self.classifier(video_feats)
        return logits, feats, video_feats

model = ResNet18VideoBinaryClassifier(
    pretrained=True,
    freeze_backbone=FREEZE_BACKBONE
).to(DEVICE)


# Loss / optimizer

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


# Helpers

def softmax_np(logits):
    x = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def compute_binary_metrics(y_true, logits_np):
    probs = softmax_np(logits_np)
    y_pred = np.argmax(logits_np, axis=1)
    y_prob_fake = probs[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob_fake))
    except Exception:
        metrics["roc_auc"] = None

    try:
        metrics["pr_auc"] = float(average_precision_score(y_true, y_prob_fake))
    except Exception:
        metrics["pr_auc"] = None

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["fpr"] = float(fp / max(fp + tn, 1))
    metrics["fnr"] = float(fn / max(fn + tp, 1))

    return metrics, probs, y_pred

def run_one_epoch(model, loader, criterion, optimizer=None, device="cuda"):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss = 0.0
    n_samples = 0

    all_logits = []
    all_labels = []

    for xb, yb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits, frame_feats, video_feats = model(xb)
        loss = criterion(logits, yb)

        if is_train:
            loss.backward()
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        bs = xb.size(0)
        running_loss += loss.item() * bs
        n_samples += bs

        all_logits.append(logits.detach().cpu().numpy())
        all_labels.append(yb.detach().cpu().numpy())

    epoch_loss = running_loss / max(n_samples, 1)

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)

    metrics, probs, y_pred = compute_binary_metrics(labels_np, logits_np)

    result = {
        "loss": float(epoch_loss),
        **metrics,
    }
    return result


# Training loop

history = []
best_val_score = -np.inf
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_result = run_one_epoch(
        model=model,
        loader=train_loader_full,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
    )

    with torch.no_grad():
        val_result = run_one_epoch(
            model=model,
            loader=val_loader_full,
            criterion=criterion,
            optimizer=None,
            device=DEVICE,
        )

    epoch_time = time.time() - epoch_start

    row = {
        "epoch": int(epoch),

        "train_loss": train_result["loss"],
        "train_accuracy": train_result["accuracy"],
        "train_precision": train_result["precision"],
        "train_recall": train_result["recall"],
        "train_f1": train_result["f1"],
        "train_balanced_accuracy": train_result["balanced_accuracy"],
        "train_roc_auc": train_result["roc_auc"],
        "train_pr_auc": train_result["pr_auc"],
        "train_fpr": train_result["fpr"],

        "val_loss": val_result["loss"],
        "val_accuracy": val_result["accuracy"],
        "val_precision": val_result["precision"],
        "val_recall": val_result["recall"],
        "val_f1": val_result["f1"],
        "val_balanced_accuracy": val_result["balanced_accuracy"],
        "val_roc_auc": val_result["roc_auc"],
        "val_pr_auc": val_result["pr_auc"],
        "val_fpr": val_result["fpr"],

        "epoch_seconds": float(epoch_time),
    }
    history.append(row)

    # same selection strategy as dev run
    val_score = (
        (val_result["balanced_accuracy"] * 1000.0)
        - (val_result["fpr"] * 10.0)
    )

    if val_score > best_val_score:
        best_val_score = val_score
        best_state = copy.deepcopy(model.state_dict())

        torch.save({
            "epoch": epoch,
            "model_state_dict": best_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_score": best_val_score,
            "history": history,
            "config": {
                "num_epochs": NUM_EPOCHS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "freeze_backbone": FREEZE_BACKBONE,
                "use_class_weight": USE_CLASS_WEIGHT,
            }
        }, BEST_MODEL_PATH)

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_score": best_val_score,
        "history": history,
    }, LAST_MODEL_PATH)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={train_result['loss']:.4f} | "
        f"train_f1={train_result['f1']:.4f} | "
        f"train_bal_acc={train_result['balanced_accuracy']:.4f} | "
        f"val_loss={val_result['loss']:.4f} | "
        f"val_f1={val_result['f1']:.4f} | "
        f"val_bal_acc={val_result['balanced_accuracy']:.4f} | "
        f"val_fpr={val_result['fpr']:.4f} | "
        f"time={epoch_time:.1f}s"
    )


# Restore best model

if best_state is not None:
    model.load_state_dict(best_state)


# Save history

with open(TRAINING_HISTORY_JSON, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

history_df = pd.DataFrame(history)

print("\nFull-run binary training finished.")
print("Best checkpoint:", BEST_MODEL_PATH)
print("Last checkpoint:", LAST_MODEL_PATH)
print("History JSON    :", TRAINING_HISTORY_JSON)

print("\nTraining history preview:")
display(history_df)

In [ ]:

# Full-run binary evaluation on train/val/test


import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


# Paths

BEST_MODEL_PATH = CHECKPOINT_ROOT / "binary_baseline_resnet18_fullrun" / "best_model.pt"
EVAL_DIR = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

METRICS_JSON = EVAL_DIR / "metrics_train_val_test.json"
PREDICTIONS_CSV = EVAL_DIR / "predictions_train_val_test.csv"
CONFUSION_JSON = EVAL_DIR / "confusion_matrices_train_val_test.json"

print("Best checkpoint:", BEST_MODEL_PATH)
print("Eval dir       :", EVAL_DIR)


# Recreate model class

class ResNet18VideoBinaryClassifier(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=False):
        super().__init__()

        from torchvision import models
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, 2)
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        video_feats = feats.mean(dim=1)
        logits = self.classifier(video_feats)
        return logits, feats, video_feats


# Helpers

def softmax_np(logits):
    x = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def compute_binary_metrics(y_true, logits_np):
    probs = softmax_np(logits_np)
    y_pred = np.argmax(logits_np, axis=1)
    y_prob_fake = probs[:, 1]

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob_fake))
    except Exception:
        metrics["roc_auc"] = None

    try:
        metrics["pr_auc"] = float(average_precision_score(y_true, y_prob_fake))
    except Exception:
        metrics["pr_auc"] = None

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["fpr"] = float(fp / max(fp + tn, 1))
    metrics["fnr"] = float(fn / max(fn + tp, 1))

    return metrics, probs, y_pred, cm

@torch.no_grad()
def evaluate_loader(model, loader, split_name, device="cuda"):
    model.eval()

    all_logits = []
    all_labels = []
    all_rows = []

    for xb, yb, metab in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits, frame_feats, video_feats = model(xb)

        logits_np = logits.detach().cpu().numpy()
        labels_np = yb.detach().cpu().numpy()
        probs = softmax_np(logits_np)
        preds = np.argmax(logits_np, axis=1)

        all_logits.append(logits_np)
        all_labels.append(labels_np)

        for i, meta in enumerate(metab):
            all_rows.append({
                "eval_split": split_name,
                "sample_id": meta["sample_id"],
                "dataset": meta["dataset"],
                "family_label_name": meta["family_label_name"],
                "binary_label_name": meta["binary_label_name"],
                "y_true": int(labels_np[i]),
                "y_pred": int(preds[i]),
                "prob_real": float(probs[i, 0]),
                "prob_fake": float(probs[i, 1]),
                "logit_real": float(logits_np[i, 0]),
                "logit_fake": float(logits_np[i, 1]),
            })

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)

    metrics, probs, y_pred, cm = compute_binary_metrics(labels_np, logits_np)
    return metrics, cm, pd.DataFrame(all_rows)


# Load checkpoint

checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

model = ResNet18VideoBinaryClassifier(
    pretrained=False,
    freeze_backbone=False
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Best val score         :", checkpoint.get("best_val_score", "unknown"))


# Evaluate all splits

train_metrics, train_cm, train_pred_df = evaluate_loader(model, train_loader_full, "train", device=DEVICE)
val_metrics, val_cm, val_pred_df = evaluate_loader(model, val_loader_full, "val", device=DEVICE)
test_metrics, test_cm, test_pred_df = evaluate_loader(model, test_loader_full, "test", device=DEVICE)

pred_df = pd.concat([train_pred_df, val_pred_df, test_pred_df], axis=0).reset_index(drop=True)
pred_df.to_csv(PREDICTIONS_CSV, index=False)

metrics_summary = {
    "train": train_metrics,
    "val": val_metrics,
    "test": test_metrics,
    "checkpoint_epoch": checkpoint.get("epoch", None),
    "best_val_score": checkpoint.get("best_val_score", None),
}

confusion_summary = {
    "train": train_cm.tolist(),
    "val": val_cm.tolist(),
    "test": test_cm.tolist(),
}

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

with open(CONFUSION_JSON, "w", encoding="utf-8") as f:
    json.dump(confusion_summary, f, indent=2)


# Print metrics

def print_metrics_block(name, m):
    print(f"\n{name.upper()} METRICS")
    print(f"  accuracy           : {m['accuracy']:.4f}")
    print(f"  precision          : {m['precision']:.4f}")
    print(f"  recall             : {m['recall']:.4f}")
    print(f"  f1                 : {m['f1']:.4f}")
    print(f"  balanced_accuracy  : {m['balanced_accuracy']:.4f}")
    print(f"  roc_auc            : {m['roc_auc']:.4f}" if m["roc_auc"] is not None else "  roc_auc            : None")
    print(f"  pr_auc             : {m['pr_auc']:.4f}" if m["pr_auc"] is not None else "  pr_auc             : None")
    print(f"  tn                 : {m['tn']}")
    print(f"  fp                 : {m['fp']}")
    print(f"  fn                 : {m['fn']}")
    print(f"  tp                 : {m['tp']}")
    print(f"  fpr                : {m['fpr']:.4f}")
    print(f"  fnr                : {m['fnr']:.4f}")

print_metrics_block("train", train_metrics)
print("Train confusion matrix:\n", train_cm)

print_metrics_block("val", val_metrics)
print("Val confusion matrix:\n", val_cm)

print_metrics_block("test", test_metrics)
print("Test confusion matrix:\n", test_cm)

print("\nPrediction counts by eval split:")
print(pred_df.groupby(["eval_split", "y_true", "y_pred"]).size())

print("\nPrediction counts by dataset and eval split:")
print(pred_df.groupby(["eval_split", "dataset", "y_true", "y_pred"]).size())

print("\nSaved:")
print("  Metrics JSON        :", METRICS_JSON)
print("  Predictions CSV     :", PREDICTIONS_CSV)
print("  Confusion JSON      :", CONFUSION_JSON)

print("\nPredictions preview:")
display(pred_df.head(20))

In [ ]:

# Full-run threshold tuning for binary model

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


# Input predictions

PREDICTIONS_CSV = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun" / "predictions_train_val_test.csv"


# Output files

THRESHOLD_DIR = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun" / "threshold_tuning"
THRESHOLD_DIR.mkdir(parents=True, exist_ok=True)

VAL_SWEEP_CSV = THRESHOLD_DIR / "validation_threshold_sweep.csv"
TEST_OPS_CSV = THRESHOLD_DIR / "test_operating_points.csv"
THRESHOLD_SUMMARY_JSON = THRESHOLD_DIR / "threshold_summary.json"

print("Predictions CSV :", PREDICTIONS_CSV)
print("Threshold dir   :", THRESHOLD_DIR)


# Load predictions

pred_df = pd.read_csv(PREDICTIONS_CSV)

val_df = pred_df[pred_df["eval_split"] == "val"].copy().reset_index(drop=True)
test_df = pred_df[pred_df["eval_split"] == "test"].copy().reset_index(drop=True)

print("\nLoaded prediction splits:")
print("  val  :", val_df.shape)
print("  test :", test_df.shape)


# Helpers

def compute_metrics_from_threshold(df, threshold):
    y_true = df["y_true"].to_numpy().astype(int)
    prob_fake = df["prob_fake"].to_numpy().astype(float)
    y_pred = (prob_fake >= threshold).astype(int)

    metrics = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, prob_fake))
    except Exception:
        metrics["roc_auc"] = None

    try:
        metrics["pr_auc"] = float(average_precision_score(y_true, prob_fake))
    except Exception:
        metrics["pr_auc"] = None

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["fpr"] = float(fp / max(fp + tn, 1))
    metrics["fnr"] = float(fn / max(fn + tp, 1))
    metrics["specificity"] = float(tn / max(tn + fp, 1))
    metrics["predicted_fake_rate"] = float(y_pred.mean())

    return metrics

def select_best_row(df, sort_cols, ascending_list):
    return df.sort_values(sort_cols, ascending=ascending_list).iloc[0]


# Threshold sweep on validation set

thresholds = np.round(np.linspace(0.01, 0.99, 99), 2)

val_rows = []
for thr in thresholds:
    val_rows.append(compute_metrics_from_threshold(val_df, thr))

val_sweep_df = pd.DataFrame(val_rows)
val_sweep_df.to_csv(VAL_SWEEP_CSV, index=False)

print("\nValidation threshold sweep preview:")
display(val_sweep_df.head(10))


# Select candidate thresholds

best_bal_row = select_best_row(
    val_sweep_df,
    sort_cols=["balanced_accuracy", "fpr", "f1"],
    ascending_list=[False, True, False]
)

best_f1_row = select_best_row(
    val_sweep_df,
    sort_cols=["f1", "balanced_accuracy", "fpr"],
    ascending_list=[False, False, True]
)

val_fpr_005 = val_sweep_df[val_sweep_df["fpr"] <= 0.05].copy()
best_fpr005_row = None
if len(val_fpr_005) > 0:
    best_fpr005_row = select_best_row(
        val_fpr_005,
        sort_cols=["balanced_accuracy", "recall", "f1"],
        ascending_list=[False, False, False]
    )

val_fpr_010 = val_sweep_df[val_sweep_df["fpr"] <= 0.10].copy()
best_fpr010_row = None
if len(val_fpr_010) > 0:
    best_fpr010_row = select_best_row(
        val_fpr_010,
        sort_cols=["balanced_accuracy", "recall", "f1"],
        ascending_list=[False, False, False]
    )

candidate_rows = []
candidate_rows.append(("best_balanced_accuracy", best_bal_row))
candidate_rows.append(("best_f1", best_f1_row))
if best_fpr005_row is not None:
    candidate_rows.append(("best_under_fpr_0.05", best_fpr005_row))
if best_fpr010_row is not None:
    candidate_rows.append(("best_under_fpr_0.10", best_fpr010_row))

candidate_summary_rows = []
for name, row in candidate_rows:
    candidate_summary_rows.append({
        "selection_name": name,
        **row.to_dict()
    })

candidate_summary_df = pd.DataFrame(candidate_summary_rows)

print("\nSelected validation operating points:")
display(candidate_summary_df)


# Evaluate selected thresholds on test set

test_rows = []
for name, row in candidate_rows:
    thr = float(row["threshold"])
    test_metrics = compute_metrics_from_threshold(test_df, thr)
    test_rows.append({
        "selection_name": name,
        **test_metrics
    })

test_ops_df = pd.DataFrame(test_rows)
test_ops_df.to_csv(TEST_OPS_CSV, index=False)

print("\nTest operating points:")
display(test_ops_df)


# Save JSON summary

summary = {
    "n_val_samples": int(len(val_df)),
    "n_test_samples": int(len(test_df)),
    "selected_thresholds": {
        row["selection_name"]: {
            "threshold": float(row["threshold"]),
            "val_accuracy": float(row["accuracy"]),
            "val_precision": float(row["precision"]),
            "val_recall": float(row["recall"]),
            "val_f1": float(row["f1"]),
            "val_balanced_accuracy": float(row["balanced_accuracy"]),
            "val_fpr": float(row["fpr"]),
            "val_fnr": float(row["fnr"]),
        }
        for _, row in candidate_summary_df.iterrows()
    },
    "test_operating_points": {
        row["selection_name"]: {
            "threshold": float(row["threshold"]),
            "test_accuracy": float(row["accuracy"]),
            "test_precision": float(row["precision"]),
            "test_recall": float(row["recall"]),
            "test_f1": float(row["f1"]),
            "test_balanced_accuracy": float(row["balanced_accuracy"]),
            "test_fpr": float(row["fpr"]),
            "test_fnr": float(row["fnr"]),
            "tn": int(row["tn"]),
            "fp": int(row["fp"]),
            "fn": int(row["fn"]),
            "tp": int(row["tp"]),
        }
        for _, row in test_ops_df.iterrows()
    }
}

with open(THRESHOLD_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Compact summary

print("\nCompact validation selections:")
for _, row in candidate_summary_df.iterrows():
    print(
        f"{row['selection_name']}: "
        f"thr={row['threshold']:.2f}, "
        f"val_bal_acc={row['balanced_accuracy']:.4f}, "
        f"val_f1={row['f1']:.4f}, "
        f"val_fpr={row['fpr']:.4f}, "
        f"val_recall={row['recall']:.4f}"
    )

print("\nCompact test operating points:")
for _, row in test_ops_df.iterrows():
    print(
        f"{row['selection_name']}: "
        f"thr={row['threshold']:.2f}, "
        f"test_bal_acc={row['balanced_accuracy']:.4f}, "
        f"test_f1={row['f1']:.4f}, "
        f"test_fpr={row['fpr']:.4f}, "
        f"test_recall={row['recall']:.4f}"
    )

print("\nSaved:")
print("  Validation sweep CSV   :", VAL_SWEEP_CSV)
print("  Test operating CSV     :", TEST_OPS_CSV)
print("  Threshold summary JSON :", THRESHOLD_SUMMARY_JSON)

In [ ]:
# Full-run temperature scaling calibration


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from scipy.optimize import minimize


# Input predictions

PREDICTIONS_CSV = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun" / "predictions_train_val_test.csv"


# Output files

CALIB_DIR = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun" / "calibration"
CALIB_DIR.mkdir(parents=True, exist_ok=True)

CALIBRATED_PREDICTIONS_CSV = CALIB_DIR / "predictions_calibrated_train_val_test.csv"
CALIBRATION_SUMMARY_JSON = CALIB_DIR / "calibration_summary.json"

print("Predictions CSV :", PREDICTIONS_CSV)
print("Calibration dir :", CALIB_DIR)


# Load predictions

pred_df = pd.read_csv(PREDICTIONS_CSV)

train_df = pred_df[pred_df["eval_split"] == "train"].copy().reset_index(drop=True)
val_df = pred_df[pred_df["eval_split"] == "val"].copy().reset_index(drop=True)
test_df = pred_df[pred_df["eval_split"] == "test"].copy().reset_index(drop=True)

print("\nLoaded splits:")
print("  train:", train_df.shape)
print("  val  :", val_df.shape)
print("  test :", test_df.shape)


# Helpers

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def compute_ece(y_true, prob_fake, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob_fake = np.asarray(prob_fake).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (prob_fake >= left) & (prob_fake <= right)
        else:
            mask = (prob_fake >= left) & (prob_fake < right)

        if mask.sum() == 0:
            continue

        conf = prob_fake[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)

    return float(ece)

def compute_binary_metrics_from_probs(y_true, prob_fake, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob_fake = np.asarray(prob_fake).astype(float)
    y_pred = (prob_fake >= threshold).astype(int)

    metrics = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        "log_loss": float(log_loss(y_true, np.clip(prob_fake, 1e-7, 1 - 1e-7))),
        "ece_10bins": float(compute_ece(y_true, prob_fake, n_bins=10)),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, prob_fake))
    except Exception:
        metrics["roc_auc"] = None

    try:
        metrics["pr_auc"] = float(average_precision_score(y_true, prob_fake))
    except Exception:
        metrics["pr_auc"] = None

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["fpr"] = float(fp / max(fp + tn, 1))
    metrics["fnr"] = float(fn / max(fn + tp, 1))

    return metrics, cm


# Learn temperature on validation logits
# Use fake logit margin = logit_fake - logit_real

val_margin = (val_df["logit_fake"] - val_df["logit_real"]).to_numpy().astype(float)
val_y = val_df["y_true"].to_numpy().astype(int)

def nll_for_temperature(log_T):
    T = np.exp(log_T[0])
    calibrated_prob = sigmoid(val_margin / T)
    return log_loss(val_y, np.clip(calibrated_prob, 1e-7, 1 - 1e-7))

opt_result = minimize(
    nll_for_temperature,
    x0=np.array([0.0], dtype=np.float64),   # log(T)=0 => T=1
    method="L-BFGS-B",
    bounds=[(-4.0, 4.0)]
)

best_log_T = float(opt_result.x[0])
best_T = float(np.exp(best_log_T))

print("\nOptimization success:", opt_result.success)
print("Learned temperature T:", best_T)


# Apply calibration to all splits

def apply_temperature_scaling(df, T):
    out = df.copy()
    margin = (out["logit_fake"] - out["logit_real"]).to_numpy().astype(float)
    prob_fake_cal = sigmoid(margin / T)
    prob_real_cal = 1.0 - prob_fake_cal

    out["prob_fake_calibrated"] = prob_fake_cal
    out["prob_real_calibrated"] = prob_real_cal
    out["temperature_T"] = T
    return out

train_cal_df = apply_temperature_scaling(train_df, best_T)
val_cal_df = apply_temperature_scaling(val_df, best_T)
test_cal_df = apply_temperature_scaling(test_df, best_T)

all_cal_df = pd.concat([train_cal_df, val_cal_df, test_cal_df], axis=0).reset_index(drop=True)
all_cal_df.to_csv(CALIBRATED_PREDICTIONS_CSV, index=False)


# Evaluate raw vs calibrated at threshold 0.50

summary = {
    "temperature_T": best_T,
    "optimization_success": bool(opt_result.success),
}

for split_name, raw_df, cal_df in [
    ("train", train_df, train_cal_df),
    ("val", val_df, val_cal_df),
    ("test", test_df, test_cal_df),
]:
    y_true = raw_df["y_true"].to_numpy().astype(int)

    raw_prob = raw_df["prob_fake"].to_numpy().astype(float)
    cal_prob = cal_df["prob_fake_calibrated"].to_numpy().astype(float)

    raw_metrics, raw_cm = compute_binary_metrics_from_probs(y_true, raw_prob, threshold=0.5)
    cal_metrics, cal_cm = compute_binary_metrics_from_probs(y_true, cal_prob, threshold=0.5)

    summary[split_name] = {
        "raw": raw_metrics,
        "calibrated": cal_metrics,
        "raw_confusion_matrix": raw_cm.tolist(),
        "calibrated_confusion_matrix": cal_cm.tolist(),
    }

with open(CALIBRATION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Compact printout

def print_block(split_name, metrics_raw, metrics_cal):
    print(f"\n{split_name.upper()} @ threshold=0.50")
    print(
        f"  RAW        | "
        f"acc={metrics_raw['accuracy']:.4f}, "
        f"f1={metrics_raw['f1']:.4f}, "
        f"bal_acc={metrics_raw['balanced_accuracy']:.4f}, "
        f"fpr={metrics_raw['fpr']:.4f}, "
        f"ece={metrics_raw['ece_10bins']:.4f}, "
        f"brier={metrics_raw['brier_score']:.4f}"
    )
    print(
        f"  CALIBRATED | "
        f"acc={metrics_cal['accuracy']:.4f}, "
        f"f1={metrics_cal['f1']:.4f}, "
        f"bal_acc={metrics_cal['balanced_accuracy']:.4f}, "
        f"fpr={metrics_cal['fpr']:.4f}, "
        f"ece={metrics_cal['ece_10bins']:.4f}, "
        f"brier={metrics_cal['brier_score']:.4f}"
    )

for split_name in ["train", "val", "test"]:
    print_block(
        split_name,
        summary[split_name]["raw"],
        summary[split_name]["calibrated"]
    )

print("\nSaved:")
print("  Calibrated predictions CSV :", CALIBRATED_PREDICTIONS_CSV)
print("  Calibration summary JSON   :", CALIBRATION_SUMMARY_JSON)

In [ ]:
# Full-run calibrated threshold tuning + uncertain mode


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


# Input calibrated predictions

CALIBRATED_PREDICTIONS_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "binary_baseline_resnet18_fullrun"
    / "calibration"
    / "predictions_calibrated_train_val_test.csv"
)


# Output files

UNCERT_DIR = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun" / "uncertainty"
UNCERT_DIR.mkdir(parents=True, exist_ok=True)

CAL_THR_SWEEP_CSV = UNCERT_DIR / "calibrated_threshold_sweep_val.csv"
CAL_TEST_OPS_CSV = UNCERT_DIR / "calibrated_test_operating_points.csv"
UNCERTAIN_RESULTS_CSV = UNCERT_DIR / "uncertain_operating_points.csv"
UNCERTAIN_SUMMARY_JSON = UNCERT_DIR / "uncertain_summary.json"

print("Calibrated predictions CSV:", CALIBRATED_PREDICTIONS_CSV)
print("Uncertainty dir          :", UNCERT_DIR)


# Load calibrated predictions

pred_df = pd.read_csv(CALIBRATED_PREDICTIONS_CSV)

val_df = pred_df[pred_df["eval_split"] == "val"].copy().reset_index(drop=True)
test_df = pred_df[pred_df["eval_split"] == "test"].copy().reset_index(drop=True)

print("\nLoaded splits:")
print("  val  :", val_df.shape)
print("  test :", test_df.shape)


# Helpers

def compute_metrics_binary_threshold(df, threshold, prob_col="prob_fake_calibrated"):
    y_true = df["y_true"].to_numpy().astype(int)
    prob_fake = df[prob_col].to_numpy().astype(float)
    y_pred = (prob_fake >= threshold).astype(int)

    metrics = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["fpr"] = float(fp / max(fp + tn, 1))
    metrics["fnr"] = float(fn / max(fn + tp, 1))
    metrics["specificity"] = float(tn / max(tn + fp, 1))
    metrics["predicted_fake_rate"] = float(y_pred.mean())

    return metrics

def compute_metrics_with_uncertainty(df, low_thr, high_thr, prob_col="prob_fake_calibrated"):
    y_true = df["y_true"].to_numpy().astype(int)
    prob_fake = df[prob_col].to_numpy().astype(float)

    decision = np.full(len(df), fill_value=-1, dtype=int)  # -1 = uncertain
    decision[prob_fake <= low_thr] = 0
    decision[prob_fake >= high_thr] = 1

    decided_mask = decision != -1
    uncertain_mask = decision == -1

    result = {
        "low_thr": float(low_thr),
        "high_thr": float(high_thr),
        "coverage": float(decided_mask.mean()),
        "uncertain_rate": float(uncertain_mask.mean()),
        "n_total": int(len(df)),
        "n_decided": int(decided_mask.sum()),
        "n_uncertain": int(uncertain_mask.sum()),
    }

    if decided_mask.sum() == 0:
        result.update({
            "accuracy_decided": None,
            "precision_decided": None,
            "recall_decided": None,
            "f1_decided": None,
            "balanced_accuracy_decided": None,
            "fpr_decided": None,
            "fnr_decided": None,
            "tn": None,
            "fp": None,
            "fn": None,
            "tp": None,
        })
        return result

    y_true_dec = y_true[decided_mask]
    y_pred_dec = decision[decided_mask]

    result["accuracy_decided"] = float(accuracy_score(y_true_dec, y_pred_dec))
    result["precision_decided"] = float(precision_score(y_true_dec, y_pred_dec, zero_division=0))
    result["recall_decided"] = float(recall_score(y_true_dec, y_pred_dec, zero_division=0))
    result["f1_decided"] = float(f1_score(y_true_dec, y_pred_dec, zero_division=0))
    result["balanced_accuracy_decided"] = float(balanced_accuracy_score(y_true_dec, y_pred_dec))

    cm = confusion_matrix(y_true_dec, y_pred_dec, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    result["tn"] = int(tn)
    result["fp"] = int(fp)
    result["fn"] = int(fn)
    result["tp"] = int(tp)
    result["fpr_decided"] = float(fp / max(fp + tn, 1))
    result["fnr_decided"] = float(fn / max(fn + tp, 1))

    return result


# 1) Re-tune calibrated single-threshold operating points

thresholds = np.round(np.linspace(0.01, 0.99, 99), 2)

val_rows = []
for thr in thresholds:
    val_rows.append(compute_metrics_binary_threshold(val_df, thr, prob_col="prob_fake_calibrated"))

val_sweep_df = pd.DataFrame(val_rows)
val_sweep_df.to_csv(CAL_THR_SWEEP_CSV, index=False)

def select_best_row(df, sort_cols, ascending_list):
    return df.sort_values(sort_cols, ascending=ascending_list).iloc[0]

best_bal_row = select_best_row(
    val_sweep_df,
    sort_cols=["balanced_accuracy", "fpr", "f1"],
    ascending_list=[False, True, False]
)

val_fpr_005 = val_sweep_df[val_sweep_df["fpr"] <= 0.05].copy()
best_fpr005_row = None
if len(val_fpr_005) > 0:
    best_fpr005_row = select_best_row(
        val_fpr_005,
        sort_cols=["balanced_accuracy", "recall", "f1"],
        ascending_list=[False, False, False]
    )

val_fpr_010 = val_sweep_df[val_sweep_df["fpr"] <= 0.10].copy()
best_fpr010_row = None
if len(val_fpr_010) > 0:
    best_fpr010_row = select_best_row(
        val_fpr_010,
        sort_cols=["balanced_accuracy", "recall", "f1"],
        ascending_list=[False, False, False]
    )

candidate_rows = [("best_balanced_accuracy_calibrated", best_bal_row)]
if best_fpr005_row is not None:
    candidate_rows.append(("best_under_fpr_0.05_calibrated", best_fpr005_row))
if best_fpr010_row is not None:
    candidate_rows.append(("best_under_fpr_0.10_calibrated", best_fpr010_row))

test_rows = []
for name, row in candidate_rows:
    thr = float(row["threshold"])
    m = compute_metrics_binary_threshold(test_df, thr, prob_col="prob_fake_calibrated")
    test_rows.append({"selection_name": name, **m})

test_ops_df = pd.DataFrame(test_rows)
test_ops_df.to_csv(CAL_TEST_OPS_CSV, index=False)

print("\nCalibrated single-threshold validation selections:")
display(pd.DataFrame([{"selection_name": n, **r.to_dict()} for n, r in candidate_rows]))

print("\nCalibrated single-threshold test operating points:")
display(test_ops_df)


# 2) Uncertain / inconclusive band search on validation

low_thresholds = np.round(np.linspace(0.10, 0.45, 8), 2)
high_thresholds = np.round(np.linspace(0.55, 0.95, 9), 2)

uncertain_val_rows = []

for low_thr in low_thresholds:
    for high_thr in high_thresholds:
        if low_thr >= high_thr:
            continue
        m = compute_metrics_with_uncertainty(val_df, low_thr, high_thr, prob_col="prob_fake_calibrated")
        uncertain_val_rows.append(m)

uncertain_val_df = pd.DataFrame(uncertain_val_rows)

usable_uncertain_val_df = uncertain_val_df[
    (uncertain_val_df["coverage"] >= 0.60) &
    (uncertain_val_df["n_decided"] > 0)
].copy()

if len(usable_uncertain_val_df) == 0:
    raise RuntimeError("No usable uncertain operating points found. Relax coverage constraints.")

best_uncertain_bal = usable_uncertain_val_df.sort_values(
    ["balanced_accuracy_decided", "fpr_decided", "coverage", "f1_decided"],
    ascending=[False, True, False, False]
).iloc[0]

usable_low_fpr = usable_uncertain_val_df[usable_uncertain_val_df["fpr_decided"] <= 0.10].copy()
best_uncertain_lowfpr = None
if len(usable_low_fpr) > 0:
    best_uncertain_lowfpr = usable_low_fpr.sort_values(
        ["balanced_accuracy_decided", "coverage", "recall_decided"],
        ascending=[False, False, False]
    ).iloc[0]

uncertain_candidates = [("best_uncertain_balanced_accuracy", best_uncertain_bal)]
if best_uncertain_lowfpr is not None:
    uncertain_candidates.append(("best_uncertain_under_fpr_0.10", best_uncertain_lowfpr))

uncertain_test_rows = []
for name, row in uncertain_candidates:
    low_thr = float(row["low_thr"])
    high_thr = float(row["high_thr"])
    m = compute_metrics_with_uncertainty(test_df, low_thr, high_thr, prob_col="prob_fake_calibrated")
    uncertain_test_rows.append({"selection_name": name, **m})

uncertain_test_df = pd.DataFrame(uncertain_test_rows)
uncertain_test_df.to_csv(UNCERTAIN_RESULTS_CSV, index=False)

print("\nUncertain-band validation selections:")
display(pd.DataFrame([{"selection_name": n, **r.to_dict()} for n, r in uncertain_candidates]))

print("\nUncertain-band test operating points:")
display(uncertain_test_df)


# Save summary JSON

summary = {
    "calibrated_single_threshold_test_operating_points": {
        row["selection_name"]: {
            "threshold": float(row["threshold"]),
            "accuracy": float(row["accuracy"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "balanced_accuracy": float(row["balanced_accuracy"]),
            "fpr": float(row["fpr"]),
            "fnr": float(row["fnr"]),
            "tn": int(row["tn"]),
            "fp": int(row["fp"]),
            "fn": int(row["fn"]),
            "tp": int(row["tp"]),
        }
        for _, row in test_ops_df.iterrows()
    },
    "uncertain_test_operating_points": {
        row["selection_name"]: {
            "low_thr": float(row["low_thr"]),
            "high_thr": float(row["high_thr"]),
            "coverage": float(row["coverage"]),
            "uncertain_rate": float(row["uncertain_rate"]),
            "accuracy_decided": None if pd.isna(row["accuracy_decided"]) else float(row["accuracy_decided"]),
            "precision_decided": None if pd.isna(row["precision_decided"]) else float(row["precision_decided"]),
            "recall_decided": None if pd.isna(row["recall_decided"]) else float(row["recall_decided"]),
            "f1_decided": None if pd.isna(row["f1_decided"]) else float(row["f1_decided"]),
            "balanced_accuracy_decided": None if pd.isna(row["balanced_accuracy_decided"]) else float(row["balanced_accuracy_decided"]),
            "fpr_decided": None if pd.isna(row["fpr_decided"]) else float(row["fpr_decided"]),
            "fnr_decided": None if pd.isna(row["fnr_decided"]) else float(row["fnr_decided"]),
            "n_total": int(row["n_total"]),
            "n_decided": int(row["n_decided"]),
            "n_uncertain": int(row["n_uncertain"]),
        }
        for _, row in uncertain_test_df.iterrows()
    }
}

with open(UNCERTAIN_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Compact printout

print("\nCompact calibrated single-threshold test points:")
for _, row in test_ops_df.iterrows():
    print(
        f"{row['selection_name']}: "
        f"thr={row['threshold']:.2f}, "
        f"bal_acc={row['balanced_accuracy']:.4f}, "
        f"f1={row['f1']:.4f}, "
        f"fpr={row['fpr']:.4f}, "
        f"recall={row['recall']:.4f}"
    )

print("\nCompact uncertain-band test points:")
for _, row in uncertain_test_df.iterrows():
    print(
        f"{row['selection_name']}: "
        f"low={row['low_thr']:.2f}, high={row['high_thr']:.2f}, "
        f"coverage={row['coverage']:.4f}, "
        f"uncertain_rate={row['uncertain_rate']:.4f}, "
        f"bal_acc_decided={row['balanced_accuracy_decided']:.4f}, "
        f"f1_decided={row['f1_decided']:.4f}, "
        f"fpr_decided={row['fpr_decided']:.4f}, "
        f"recall_decided={row['recall_decided']:.4f}"
    )

print("\nSaved:")
print("  Calibrated threshold sweep CSV :", CAL_THR_SWEEP_CSV)
print("  Calibrated test ops CSV        :", CAL_TEST_OPS_CSV)
print("  Uncertain results CSV          :", UNCERTAIN_RESULTS_CSV)
print("  Uncertain summary JSON         :", UNCERTAIN_SUMMARY_JSON)

In [ ]:
#  Full-run binary forensic report generation


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path


# Inputs

MODEL_READY_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"
CALIBRATED_PREDICTIONS_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "binary_baseline_resnet18_fullrun"
    / "calibration"
    / "predictions_calibrated_train_val_test.csv"
)


# CHANGE THIS IF NEEDED
# Chosen decision settings from full-run results

SINGLE_THRESHOLD = 0.75
UNCERTAIN_LOW_THR = 0.15
UNCERTAIN_HIGH_THR = 0.90


# Output files

REPORT_DIR = OUTPUT_ROOT / "reports" / "binary_baseline_resnet18_fullrun"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FORENSIC_REPORT_CSV = REPORT_DIR / "forensic_report_table.csv"
FORENSIC_REPORT_TEST_PREVIEW_CSV = REPORT_DIR / "forensic_report_preview_test.csv"
FORENSIC_REPORT_SUMMARY_JSON = REPORT_DIR / "forensic_report_summary.json"

print("Model-ready CSV         :", MODEL_READY_CSV)
print("Calibrated predictions  :", CALIBRATED_PREDICTIONS_CSV)
print("Report dir              :", REPORT_DIR)


# Load files

model_ready_df = pd.read_csv(MODEL_READY_CSV, low_memory=False)
pred_df = pd.read_csv(CALIBRATED_PREDICTIONS_CSV)

print("\nLoaded:")
print("  model_ready_df:", model_ready_df.shape)
print("  pred_df       :", pred_df.shape)


# Build one metadata row per sample_id

sample_meta_df = (
    model_ready_df.groupby("sample_id", as_index=False)
    .agg(
        split=("split", "first"),
        dataset=("dataset", "first"),
        source_subset=("source_subset", "first"),
        family_label_name=("family_label_name", "first"),
        binary_label_name=("binary_label_name", "first"),
        relative_path=("relative_path", "first"),
        file_name=("file_name", "first"),
        file_path=("file_path", "first"),
        group_id=("group_id", "first"),
        identity_primary=("identity_primary", "first"),
        identity_secondary=("identity_secondary", "first"),
        n_frames=("frame_index_local", "count"),
        mean_face_confidence=("face_confidence", "mean"),
        mean_face_img_mean=("face_img_mean", "mean"),
        mean_face_img_std=("face_img_std", "mean"),
        any_too_dark=("face_is_too_dark", "max"),
        any_too_bright=("face_is_too_bright", "max"),
        any_low_contrast=("face_is_low_contrast", "max"),
    )
)


# Merge predictions + metadata

report_df = pred_df.merge(
    sample_meta_df,
    on="sample_id",
    how="left",
    suffixes=("_pred", "_meta"),
    validate="one_to_one"
)

def coalesce_columns(df, base_name, left_name=None, right_name=None):
    left_name = left_name or f"{base_name}_pred"
    right_name = right_name or f"{base_name}_meta"

    if left_name in df.columns and right_name in df.columns:
        df[base_name] = df[left_name].combine_first(df[right_name])
    elif left_name in df.columns:
        df[base_name] = df[left_name]
    elif right_name in df.columns:
        df[base_name] = df[right_name]
    return df

for col in ["eval_split", "split", "dataset", "family_label_name", "binary_label_name"]:
    report_df = coalesce_columns(report_df, col)

if "eval_split" not in report_df.columns and "split" in report_df.columns:
    report_df["eval_split"] = report_df["split"]


# Decision helpers

def decision_single_threshold(prob_fake, thr):
    return "fake" if prob_fake >= thr else "real"

def decision_uncertain_band(prob_fake, low_thr, high_thr):
    if prob_fake <= low_thr:
        return "real"
    elif prob_fake >= high_thr:
        return "fake"
    else:
        return "uncertain"

def confidence_bucket(prob_fake):
    confidence = abs(prob_fake - 0.5) * 2.0
    if confidence >= 0.80:
        return "very_high"
    elif confidence >= 0.60:
        return "high"
    elif confidence >= 0.40:
        return "moderate"
    else:
        return "low"

def provenance_status_from_metadata(row):
    file_path = str(row.get("file_path", ""))
    file_name = str(row.get("file_name", ""))

    if len(file_path) > 0 and len(file_name) > 0:
        return "available"
    return "unavailable"

def explanation_stub_text(row):
    prob_fake = float(row["prob_fake_calibrated"])
    single_decision = row["pred_auth_single_threshold"]
    uncertain_decision = row["pred_auth_uncertain_mode"]

    if uncertain_decision == "uncertain":
        return (
            "The sample falls within the calibrated inconclusive region under the abstention-aware "
            "forensic policy. Automated judgment is intentionally withheld and manual review is recommended."
        )

    if single_decision == "fake":
        return (
            f"The calibrated model indicates manipulated content with fake probability {prob_fake:.3f}. "
            "This output should be treated as forensic decision support and reviewed alongside provenance, "
            "visual artifacts, and contextual evidence."
        )

    return (
        f"The calibrated model indicates authentic content with real probability {1.0 - prob_fake:.3f}. "
        "This output should be interpreted jointly with provenance metadata and human review."
    )


# Add report fields

report_df["prob_fake_calibrated"] = report_df["prob_fake_calibrated"].astype(float)
report_df["prob_real_calibrated"] = report_df["prob_real_calibrated"].astype(float)

report_df["pred_auth_single_threshold"] = report_df["prob_fake_calibrated"].apply(
    lambda p: decision_single_threshold(p, SINGLE_THRESHOLD)
)

report_df["pred_auth_uncertain_mode"] = report_df["prob_fake_calibrated"].apply(
    lambda p: decision_uncertain_band(p, UNCERTAIN_LOW_THR, UNCERTAIN_HIGH_THR)
)

report_df["confidence_score"] = report_df["prob_fake_calibrated"].apply(
    lambda p: float(abs(p - 0.5) * 2.0)
)

report_df["confidence_bucket"] = report_df["prob_fake_calibrated"].apply(confidence_bucket)
report_df["provenance_status"] = report_df.apply(provenance_status_from_metadata, axis=1)

report_df["predicted_forgery_family_placeholder"] = report_df.apply(
    lambda row: "not_applicable_real_prediction"
    if row["pred_auth_uncertain_mode"] == "real"
    else ("unknown_fake_family" if row["pred_auth_uncertain_mode"] == "fake" else "inconclusive"),
    axis=1
)

report_df["report_explanation_text"] = report_df.apply(explanation_stub_text, axis=1)

report_df["model_name"] = "binary_baseline_resnet18_fullrun"
report_df["calibration_method"] = "temperature_scaling"
report_df["decision_mode_single_threshold"] = SINGLE_THRESHOLD
report_df["decision_mode_uncertain_low"] = UNCERTAIN_LOW_THR
report_df["decision_mode_uncertain_high"] = UNCERTAIN_HIGH_THR


# Reorder columns

final_cols = [
    "eval_split",
    "sample_id",
    "dataset",
    "source_subset",
    "file_name",
    "file_path",
    "relative_path",
    "group_id",
    "identity_primary",
    "identity_secondary",
    "binary_label_name",
    "family_label_name",
    "y_true",
    "y_pred",
    "prob_real",
    "prob_fake",
    "prob_real_calibrated",
    "prob_fake_calibrated",
    "pred_auth_single_threshold",
    "pred_auth_uncertain_mode",
    "confidence_score",
    "confidence_bucket",
    "predicted_forgery_family_placeholder",
    "provenance_status",
    "n_frames",
    "mean_face_confidence",
    "mean_face_img_mean",
    "mean_face_img_std",
    "any_too_dark",
    "any_too_bright",
    "any_low_contrast",
    "model_name",
    "calibration_method",
    "decision_mode_single_threshold",
    "decision_mode_uncertain_low",
    "decision_mode_uncertain_high",
    "report_explanation_text",
]

final_cols = [c for c in final_cols if c in report_df.columns]
report_df = report_df[final_cols].copy()


# Save files

report_df.to_csv(FORENSIC_REPORT_CSV, index=False)
report_df[report_df["eval_split"] == "test"].to_csv(FORENSIC_REPORT_TEST_PREVIEW_CSV, index=False)


# Summary

summary = {
    "n_total_reports": int(len(report_df)),
    "n_test_reports": int((report_df["eval_split"] == "test").sum()),
    "single_threshold": float(SINGLE_THRESHOLD),
    "uncertain_low_thr": float(UNCERTAIN_LOW_THR),
    "uncertain_high_thr": float(UNCERTAIN_HIGH_THR),
    "single_mode_prediction_counts": {
        str(k): int(v)
        for k, v in report_df["pred_auth_single_threshold"].value_counts().to_dict().items()
    },
    "uncertain_mode_prediction_counts": {
        str(k): int(v)
        for k, v in report_df["pred_auth_uncertain_mode"].value_counts().to_dict().items()
    },
    "uncertain_mode_counts_by_split": {
        f"{k[0]}__{k[1]}": int(v)
        for k, v in report_df.groupby(["eval_split", "pred_auth_uncertain_mode"]).size().to_dict().items()
    },
    "provenance_status_counts": {
        str(k): int(v)
        for k, v in report_df["provenance_status"].value_counts().to_dict().items()
    },
    "confidence_bucket_counts": {
        str(k): int(v)
        for k, v in report_df["confidence_bucket"].value_counts().to_dict().items()
    },
}

with open(FORENSIC_REPORT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nForensic report summary:")
print("  Total reports          :", len(report_df))
print("  Test reports           :", int((report_df["eval_split"] == "test").sum()))

print("\nSingle-threshold decision counts:")
print(report_df["pred_auth_single_threshold"].value_counts())

print("\nUncertain-mode decision counts:")
print(report_df["pred_auth_uncertain_mode"].value_counts())

print("\nUncertain-mode counts by split:")
print(report_df.groupby(["eval_split", "pred_auth_uncertain_mode"]).size())

print("\nConfidence bucket counts:")
print(report_df["confidence_bucket"].value_counts())

print("\nTest report preview:")
display(report_df[report_df["eval_split"] == "test"].head(20))

print("\nSaved:")
print("  Forensic report CSV       :", FORENSIC_REPORT_CSV)
print("  Test preview CSV          :", FORENSIC_REPORT_TEST_PREVIEW_CSV)
print("  Report summary JSON       :", FORENSIC_REPORT_SUMMARY_JSON)

In [ ]:

# Full-run binary figure generation


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, auc


# Inputs

PREDICTIONS_CSV = OUTPUT_ROOT / "evaluation" / "binary_baseline_resnet18_fullrun" / "predictions_train_val_test.csv"
CALIBRATED_PREDICTIONS_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "binary_baseline_resnet18_fullrun"
    / "calibration"
    / "predictions_calibrated_train_val_test.csv"
)
VAL_SWEEP_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "binary_baseline_resnet18_fullrun"
    / "threshold_tuning"
    / "validation_threshold_sweep.csv"
)
UNCERTAIN_RESULTS_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "binary_baseline_resnet18_fullrun"
    / "uncertainty"
    / "uncertain_operating_points.csv"
)


# CHANGE THIS IF NEEDED
# Selected operating points from full run

RAW_THRESHOLD_BALANCED = 0.69
RAW_THRESHOLD_FORENSIC = 0.85
UNCERTAIN_LOW_THR = 0.15
UNCERTAIN_HIGH_THR = 0.90


# Output figure directory

FIG_DIR = FIGURE_ROOT / "binary_baseline_resnet18_fullrun"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_CM_DEFAULT = FIG_DIR / "confusion_matrix_test_default.png"
FIG_CM_BALANCED = FIG_DIR / "confusion_matrix_test_balanced_threshold.png"
FIG_CM_FORENSIC = FIG_DIR / "confusion_matrix_test_forensic_threshold.png"
FIG_ROC = FIG_DIR / "roc_curve_test.png"
FIG_PR = FIG_DIR / "pr_curve_test.png"
FIG_THRESHOLD_SWEEP = FIG_DIR / "validation_threshold_sweep.png"
FIG_CONF_HIST = FIG_DIR / "test_probability_histogram_calibrated.png"
FIG_UNCERT_BAR = FIG_DIR / "uncertainty_coverage_barplot.png"
FIG_MANIFEST = FIG_DIR / "figure_manifest.json"

print("Figure directory:", FIG_DIR)


# Load data

pred_df = pd.read_csv(PREDICTIONS_CSV)
cal_df = pd.read_csv(CALIBRATED_PREDICTIONS_CSV)
val_sweep_df = pd.read_csv(VAL_SWEEP_CSV)
uncert_df = pd.read_csv(UNCERTAIN_RESULTS_CSV)

test_raw_df = pred_df[pred_df["eval_split"] == "test"].copy().reset_index(drop=True)
test_cal_df = cal_df[cal_df["eval_split"] == "test"].copy().reset_index(drop=True)

print("\nLoaded:")
print("  test_raw_df :", test_raw_df.shape)
print("  test_cal_df :", test_cal_df.shape)
print("  val_sweep   :", val_sweep_df.shape)
print("  uncertain   :", uncert_df.shape)


# Helpers

def save_confusion_matrix_figure(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(cm, interpolation="nearest")
    plt.colorbar(im, ax=ax)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=11
            )

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def preds_from_threshold(prob_fake, thr):
    return (np.asarray(prob_fake) >= thr).astype(int)


# Confusion matrices

y_true_test = test_cal_df["y_true"].to_numpy().astype(int)
prob_fake_cal = test_cal_df["prob_fake_calibrated"].to_numpy().astype(float)

# default 0.50
y_pred_default = preds_from_threshold(prob_fake_cal, 0.50)
cm_default = confusion_matrix(y_true_test, y_pred_default, labels=[0, 1])
save_confusion_matrix_figure(
    cm_default, ["real", "fake"],
    "Test Confusion Matrix (Calibrated, threshold=0.50)",
    FIG_CM_DEFAULT
)

# balanced threshold
y_pred_balanced = preds_from_threshold(prob_fake_cal, RAW_THRESHOLD_BALANCED)
cm_balanced = confusion_matrix(y_true_test, y_pred_balanced, labels=[0, 1])
save_confusion_matrix_figure(
    cm_balanced, ["real", "fake"],
    f"Test Confusion Matrix (threshold={RAW_THRESHOLD_BALANCED:.2f})",
    FIG_CM_BALANCED
)

# forensic threshold
y_pred_forensic = preds_from_threshold(prob_fake_cal, RAW_THRESHOLD_FORENSIC)
cm_forensic = confusion_matrix(y_true_test, y_pred_forensic, labels=[0, 1])
save_confusion_matrix_figure(
    cm_forensic, ["real", "fake"],
    f"Test Confusion Matrix (threshold={RAW_THRESHOLD_FORENSIC:.2f})",
    FIG_CM_FORENSIC
)


# ROC

fpr, tpr, _ = roc_curve(y_true_test, prob_fake_cal)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"ROC AUC = {roc_auc_val:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Full-Run Binary Test ROC Curve")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_ROC, dpi=300, bbox_inches="tight")
plt.close(fig)


# PR

precision, recall, _ = precision_recall_curve(y_true_test, prob_fake_cal)
pr_auc_val = auc(recall, precision)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision, label=f"PR AUC = {pr_auc_val:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Full-Run Binary Test Precision-Recall Curve")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(FIG_PR, dpi=300, bbox_inches="tight")
plt.close(fig)


# Validation threshold sweep

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(val_sweep_df["threshold"], val_sweep_df["balanced_accuracy"], label="Balanced Accuracy")
ax.plot(val_sweep_df["threshold"], val_sweep_df["f1"], label="F1")
ax.plot(val_sweep_df["threshold"], val_sweep_df["recall"], label="Recall")
ax.plot(val_sweep_df["threshold"], val_sweep_df["fpr"], label="FPR")
ax.axvline(RAW_THRESHOLD_BALANCED, linestyle="--", label=f"Balanced thr = {RAW_THRESHOLD_BALANCED:.2f}")
ax.axvline(RAW_THRESHOLD_FORENSIC, linestyle="--", label=f"Forensic thr = {RAW_THRESHOLD_FORENSIC:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Metric value")
ax.set_title("Validation Threshold Sweep (Full Run)")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIG_THRESHOLD_SWEEP, dpi=300, bbox_inches="tight")
plt.close(fig)


# Probability histogram

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.hist(
    test_cal_df.loc[test_cal_df["y_true"] == 0, "prob_fake_calibrated"],
    bins=25,
    alpha=0.7,
    label="True real"
)
ax.hist(
    test_cal_df.loc[test_cal_df["y_true"] == 1, "prob_fake_calibrated"],
    bins=25,
    alpha=0.7,
    label="True fake"
)
ax.axvline(0.50, linestyle="--", label="Default threshold = 0.50")
ax.axvline(RAW_THRESHOLD_BALANCED, linestyle="--", label=f"Balanced threshold = {RAW_THRESHOLD_BALANCED:.2f}")
ax.axvline(RAW_THRESHOLD_FORENSIC, linestyle="--", label=f"Forensic threshold = {RAW_THRESHOLD_FORENSIC:.2f}")
ax.axvline(UNCERTAIN_LOW_THR, linestyle=":", label=f"Uncertain low = {UNCERTAIN_LOW_THR:.2f}")
ax.axvline(UNCERTAIN_HIGH_THR, linestyle=":", label=f"Uncertain high = {UNCERTAIN_HIGH_THR:.2f}")
ax.set_xlabel("Calibrated fake probability")
ax.set_ylabel("Number of samples")
ax.set_title("Full-Run Binary Test Probability Distribution")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIG_CONF_HIST, dpi=300, bbox_inches="tight")
plt.close(fig)


# Uncertainty coverage plot

fig, ax = plt.subplots(figsize=(7, 5))
labels = uncert_df["selection_name"].tolist()
x = np.arange(len(labels))
ax.bar(x - 0.15, uncert_df["coverage"], width=0.30, label="Coverage")
ax.bar(x + 0.15, uncert_df["uncertain_rate"], width=0.30, label="Uncertain rate")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Proportion")
ax.set_title("Abstention-Aware Operating Modes")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIG_UNCERT_BAR, dpi=300, bbox_inches="tight")
plt.close(fig)


# Save manifest

manifest = {
    "confusion_default": str(FIG_CM_DEFAULT),
    "confusion_balanced": str(FIG_CM_BALANCED),
    "confusion_forensic": str(FIG_CM_FORENSIC),
    "roc_curve": str(FIG_ROC),
    "pr_curve": str(FIG_PR),
    "threshold_sweep": str(FIG_THRESHOLD_SWEEP),
    "confidence_histogram": str(FIG_CONF_HIST),
    "uncertainty_coverage_plot": str(FIG_UNCERT_BAR),
    "balanced_threshold": float(RAW_THRESHOLD_BALANCED),
    "forensic_threshold": float(RAW_THRESHOLD_FORENSIC),
    "uncertain_low_threshold": float(UNCERTAIN_LOW_THR),
    "uncertain_high_threshold": float(UNCERTAIN_HIGH_THR),
    "test_roc_auc": float(roc_auc_val),
    "test_pr_auc": float(pr_auc_val),
}

with open(FIG_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved figures:")
print("  Confusion default     :", FIG_CM_DEFAULT)
print("  Confusion balanced    :", FIG_CM_BALANCED)
print("  Confusion forensic    :", FIG_CM_FORENSIC)
print("  ROC curve             :", FIG_ROC)
print("  PR curve              :", FIG_PR)
print("  Threshold sweep       :", FIG_THRESHOLD_SWEEP)
print("  Probability histogram :", FIG_CONF_HIST)
print("  Uncertainty coverage  :", FIG_UNCERT_BAR)
print("  Figure manifest       :", FIG_MANIFEST)

In [ ]:

# Full-run multiclass dataset and dataloaders


import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# This cell assumes already defined:
# - METADATA_ROOT
# - EXPERIMENT_RUN_DIR
# - FACE_IMAGE_SIZE
# - MULTICLASS_LABELS



# CHANGE THIS IF NEEDED
# Input model-ready manifest

MODEL_READY_CSV_FULL = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"


# CHANGE THIS IF NEEDED
# Data loading settings

FRAMES_PER_SAMPLE = 8
IMAGE_SIZE = FACE_IMAGE_SIZE
BATCH_SIZE = 8
NUM_WORKERS = 4 if os.name != "nt" else 0
PIN_MEMORY = torch.cuda.is_available()

# Optional staged debugging
# Set to None to use all full-run samples
MAX_TRAIN_SAMPLES_MC = None
MAX_VAL_SAMPLES_MC = None
MAX_TEST_SAMPLES_MC = None


# Load manifest

model_df = pd.read_csv(MODEL_READY_CSV_FULL, low_memory=False)

print("Loaded full model-ready manifest:", model_df.shape)

required_cols = [
    "sample_id", "split", "dataset", "family_label_name",
    "frame_index_local", "face_crop_path"
]
missing_cols = [c for c in required_cols if c not in model_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")


# Attach multiclass labels

model_df = model_df.copy()
model_df["multiclass_label"] = model_df["family_label_name"].map(MULTICLASS_LABELS)

if model_df["multiclass_label"].isna().any():
    bad = model_df.loc[model_df["multiclass_label"].isna(), "family_label_name"].unique().tolist()
    raise ValueError(f"Unmapped family labels found: {bad}")

model_df["multiclass_label"] = model_df["multiclass_label"].astype(int)

print("\nFamily label distribution (frame rows):")
print(model_df["family_label_name"].value_counts())


# Group rows into one video sample = 8 face crops

group_keys = [
    "sample_id", "split", "dataset",
    "family_label_name", "multiclass_label"
]

sample_groups = []

for key_vals, sub_df in model_df.groupby(group_keys):
    sub_df = sub_df.sort_values("frame_index_local").copy()

    frame_paths = sub_df["face_crop_path"].tolist()
    frame_indices = sub_df["frame_index_local"].tolist()

    if len(frame_paths) != FRAMES_PER_SAMPLE:
        continue
    if frame_indices != list(range(FRAMES_PER_SAMPLE)):
        continue

    sample_groups.append({
        "sample_id": key_vals[0],
        "split": key_vals[1],
        "dataset": key_vals[2],
        "family_label_name": key_vals[3],
        "multiclass_label": int(key_vals[4]),
        "frame_paths": frame_paths,
    })

samples_mc_df = pd.DataFrame(sample_groups)

print("\nGrouped video-level samples:", samples_mc_df.shape)


# Split tables

train_samples_mc_full = samples_mc_df[samples_mc_df["split"] == "train"].reset_index(drop=True)
val_samples_mc_full = samples_mc_df[samples_mc_df["split"] == "val"].reset_index(drop=True)
test_samples_mc_full = samples_mc_df[samples_mc_df["split"] == "test"].reset_index(drop=True)

if MAX_TRAIN_SAMPLES_MC is not None:
    train_samples_mc_full = train_samples_mc_full.head(MAX_TRAIN_SAMPLES_MC).copy()
if MAX_VAL_SAMPLES_MC is not None:
    val_samples_mc_full = val_samples_mc_full.head(MAX_VAL_SAMPLES_MC).copy()
if MAX_TEST_SAMPLES_MC is not None:
    test_samples_mc_full = test_samples_mc_full.head(MAX_TEST_SAMPLES_MC).copy()

print("\nVideo-level sample counts:")
print("  train:", len(train_samples_mc_full))
print("  val  :", len(val_samples_mc_full))
print("  test :", len(test_samples_mc_full))

print("\nFamily distribution by split:")
print(samples_mc_df.groupby(["split", "family_label_name"]).size())

print("\nDataset distribution by split:")
print(samples_mc_df.groupby(["split", "dataset"]).size())


# Transforms

train_transform_mc_full = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform_mc_full = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# Dataset

class FullMulticlassVideoFaceDataset(Dataset):
    def __init__(self, sample_table: pd.DataFrame, transform=None):
        self.sample_table = sample_table.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.sample_table)

    def __getitem__(self, idx):
        row = self.sample_table.iloc[idx]

        frame_tensors = []
        for frame_path in row["frame_paths"]:
            img = Image.open(frame_path).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
            frame_tensors.append(img)

        x = torch.stack(frame_tensors, dim=0)  # [T, C, H, W]
        y = torch.tensor(int(row["multiclass_label"]), dtype=torch.long)

        meta = {
            "sample_id": row["sample_id"],
            "split": row["split"],
            "dataset": row["dataset"],
            "family_label_name": row["family_label_name"],
            "multiclass_label": int(row["multiclass_label"]),
        }

        return x, y, meta


# Datasets

train_dataset_mc_full = FullMulticlassVideoFaceDataset(train_samples_mc_full, transform=train_transform_mc_full)
val_dataset_mc_full = FullMulticlassVideoFaceDataset(val_samples_mc_full, transform=eval_transform_mc_full)
test_dataset_mc_full = FullMulticlassVideoFaceDataset(test_samples_mc_full, transform=eval_transform_mc_full)


# Custom collate

def custom_collate_mc_full(batch):
    xs, ys, metas = zip(*batch)
    xs = torch.stack(xs, dim=0)   # [B, T, C, H, W]
    ys = torch.stack(ys, dim=0)   # [B]
    metas = list(metas)
    return xs, ys, metas


# Dataloaders

train_loader_mc_full = DataLoader(
    train_dataset_mc_full,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=custom_collate_mc_full,
)

val_loader_mc_full = DataLoader(
    val_dataset_mc_full,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=custom_collate_mc_full,
)

test_loader_mc_full = DataLoader(
    test_dataset_mc_full,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=custom_collate_mc_full,
)


# Sanity check batch

xb, yb, metab = next(iter(train_loader_mc_full))

print("\nSanity check batch:")
print("  x shape:", tuple(xb.shape))
print("  y shape:", tuple(yb.shape))
print("  y dtype:", yb.dtype)
print("  batch label counts:", Counter(yb.cpu().numpy().tolist()))
print("  first meta:", metab[0])


# Save loader summary

loader_summary_mc = {
    "n_train_samples": int(len(train_dataset_mc_full)),
    "n_val_samples": int(len(val_dataset_mc_full)),
    "n_test_samples": int(len(test_dataset_mc_full)),
    "frames_per_sample": int(FRAMES_PER_SAMPLE),
    "image_size": int(IMAGE_SIZE),
    "batch_size": int(BATCH_SIZE),
    "num_workers": int(NUM_WORKERS),
    "pin_memory": bool(PIN_MEMORY),
    "train_family_distribution": {
        str(k): int(v) for k, v in train_samples_mc_full["family_label_name"].value_counts().to_dict().items()
    },
    "val_family_distribution": {
        str(k): int(v) for k, v in val_samples_mc_full["family_label_name"].value_counts().to_dict().items()
    },
    "test_family_distribution": {
        str(k): int(v) for k, v in test_samples_mc_full["family_label_name"].value_counts().to_dict().items()
    },
}

loader_summary_json_mc = EXPERIMENT_RUN_DIR / "multiclass_fullrun_dataset_loader_summary.json"
with open(loader_summary_json_mc, "w", encoding="utf-8") as f:
    json.dump(loader_summary_mc, f, indent=2)

print("\nSaved:")
print("  Multiclass loader summary JSON :", loader_summary_json_mc)

In [ ]:

# Full-run multiclass baseline training


import os
import json
import copy
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


#  training configuration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES_MC = len(MULTICLASS_LABELS)
IDX_TO_CLASS = {v: k for k, v in MULTICLASS_LABELS.items()}

NUM_EPOCHS_MC = 10
LEARNING_RATE_MC = 1e-4
WEIGHT_DECAY_MC = 1e-4
USE_CLASS_WEIGHT_MC = True
MAX_GRAD_NORM_MC = 1.0
FREEZE_BACKBONE_MC = False


# Output paths

CHECKPOINT_DIR_MC = CHECKPOINT_ROOT / "multiclass_baseline_resnet18_fullrun"
CHECKPOINT_DIR_MC.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH_MC = CHECKPOINT_DIR_MC / "best_model.pt"
LAST_MODEL_PATH_MC = CHECKPOINT_DIR_MC / "last_model.pt"
TRAINING_HISTORY_JSON_MC = CHECKPOINT_DIR_MC / "training_history.json"

print("Using device:", DEVICE)
print("Num classes :", NUM_CLASSES_MC)


# Compute multiclass weights

train_label_counts_mc = (
    train_samples_mc_full["multiclass_label"]
    .value_counts()
    .sort_index()
    .to_dict()
)

print("\nTrain multiclass counts:")
for cls_idx in range(NUM_CLASSES_MC):
    print(f"  class {cls_idx} ({IDX_TO_CLASS[cls_idx]}): {train_label_counts_mc.get(cls_idx, 0)}")

if USE_CLASS_WEIGHT_MC:
    total_train = sum(train_label_counts_mc.values())
    class_weights_mc = []
    for cls_idx in range(NUM_CLASSES_MC):
        count = train_label_counts_mc.get(cls_idx, 0)
        class_weights_mc.append(total_train / max(count, 1))

    class_weights_mc = torch.tensor(class_weights_mc, dtype=torch.float32, device=DEVICE)
    print("\nUsing multiclass weights:", class_weights_mc.detach().cpu().numpy())
else:
    class_weights_mc = None
    print("\nMulticlass class weighting disabled")


# Model

class ResNet18VideoMulticlassClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_backbone=False):
        super().__init__()

        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        video_feats = feats.mean(dim=1)
        logits = self.classifier(video_feats)
        return logits, feats, video_feats

model_mc = ResNet18VideoMulticlassClassifier(
    num_classes=NUM_CLASSES_MC,
    pretrained=True,
    freeze_backbone=FREEZE_BACKBONE_MC
).to(DEVICE)


# Loss / optimizer

criterion_mc = nn.CrossEntropyLoss(weight=class_weights_mc)
optimizer_mc = optim.AdamW(model_mc.parameters(), lr=LEARNING_RATE_MC, weight_decay=WEIGHT_DECAY_MC)


# Helpers

def compute_multiclass_metrics(y_true, logits_np, num_classes):
    y_pred = np.argmax(logits_np, axis=1)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    return metrics, cm

def run_one_epoch_mc(model, loader, criterion, num_classes, optimizer=None, device="cuda"):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss = 0.0
    n_samples = 0

    all_logits = []
    all_labels = []

    for xb, yb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits, frame_feats, video_feats = model(xb)
        loss = criterion(logits, yb)

        if is_train:
            loss.backward()
            if MAX_GRAD_NORM_MC is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM_MC)
            optimizer.step()

        bs = xb.size(0)
        running_loss += loss.item() * bs
        n_samples += bs

        all_logits.append(logits.detach().cpu().numpy())
        all_labels.append(yb.detach().cpu().numpy())

    epoch_loss = running_loss / max(n_samples, 1)

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)

    metrics, cm = compute_multiclass_metrics(labels_np, logits_np, num_classes=num_classes)

    result = {
        "loss": float(epoch_loss),
        **metrics,
    }
    return result, cm


# Training loop

history_mc = []
best_val_score_mc = -np.inf
best_state_mc = None

for epoch in range(1, NUM_EPOCHS_MC + 1):
    epoch_start = time.time()

    train_result_mc, train_cm_mc = run_one_epoch_mc(
        model=model_mc,
        loader=train_loader_mc_full,
        criterion=criterion_mc,
        num_classes=NUM_CLASSES_MC,
        optimizer=optimizer_mc,
        device=DEVICE,
    )

    with torch.no_grad():
        val_result_mc, val_cm_mc = run_one_epoch_mc(
            model=model_mc,
            loader=val_loader_mc_full,
            criterion=criterion_mc,
            num_classes=NUM_CLASSES_MC,
            optimizer=None,
            device=DEVICE,
        )

    epoch_time = time.time() - epoch_start

    row = {
        "epoch": int(epoch),

        "train_loss": train_result_mc["loss"],
        "train_accuracy": train_result_mc["accuracy"],
        "train_macro_f1": train_result_mc["macro_f1"],
        "train_weighted_f1": train_result_mc["weighted_f1"],
        "train_balanced_accuracy": train_result_mc["balanced_accuracy"],

        "val_loss": val_result_mc["loss"],
        "val_accuracy": val_result_mc["accuracy"],
        "val_macro_f1": val_result_mc["macro_f1"],
        "val_weighted_f1": val_result_mc["weighted_f1"],
        "val_balanced_accuracy": val_result_mc["balanced_accuracy"],

        "epoch_seconds": float(epoch_time),
    }
    history_mc.append(row)

    # prioritize macro-f1 and balanced accuracy for family attribution
    val_score_mc = (
        (val_result_mc["macro_f1"] * 1000.0)
        + (val_result_mc["balanced_accuracy"] * 100.0)
    )

    if val_score_mc > best_val_score_mc:
        best_val_score_mc = val_score_mc
        best_state_mc = copy.deepcopy(model_mc.state_dict())

        torch.save({
            "epoch": epoch,
            "model_state_dict": best_state_mc,
            "optimizer_state_dict": optimizer_mc.state_dict(),
            "best_val_score": best_val_score_mc,
            "history": history_mc,
            "config": {
                "num_epochs": NUM_EPOCHS_MC,
                "learning_rate": LEARNING_RATE_MC,
                "weight_decay": WEIGHT_DECAY_MC,
                "freeze_backbone": FREEZE_BACKBONE_MC,
                "use_class_weight": USE_CLASS_WEIGHT_MC,
                "num_classes": NUM_CLASSES_MC,
            }
        }, BEST_MODEL_PATH_MC)

    torch.save({
        "epoch": epoch,
        "model_state_dict": model_mc.state_dict(),
        "optimizer_state_dict": optimizer_mc.state_dict(),
        "best_val_score": best_val_score_mc,
        "history": history_mc,
    }, LAST_MODEL_PATH_MC)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS_MC} | "
        f"train_loss={train_result_mc['loss']:.4f} | "
        f"train_macro_f1={train_result_mc['macro_f1']:.4f} | "
        f"train_bal_acc={train_result_mc['balanced_accuracy']:.4f} | "
        f"val_loss={val_result_mc['loss']:.4f} | "
        f"val_macro_f1={val_result_mc['macro_f1']:.4f} | "
        f"val_bal_acc={val_result_mc['balanced_accuracy']:.4f} | "
        f"time={epoch_time:.1f}s"
    )


# Restore best model

if best_state_mc is not None:
    model_mc.load_state_dict(best_state_mc)


# Save history

with open(TRAINING_HISTORY_JSON_MC, "w", encoding="utf-8") as f:
    json.dump(history_mc, f, indent=2)

history_df_mc = pd.DataFrame(history_mc)

print("\nFull-run multiclass training finished.")
print("Best checkpoint:", BEST_MODEL_PATH_MC)
print("Last checkpoint:", LAST_MODEL_PATH_MC)
print("History JSON    :", TRAINING_HISTORY_JSON_MC)

print("\nTraining history preview:")
display(history_df_mc)

In [ ]:

# Full-run multiclass evaluation on train/val/test


import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


# Paths

BEST_MODEL_PATH_MC = CHECKPOINT_ROOT / "multiclass_baseline_resnet18_fullrun" / "best_model.pt"
EVAL_DIR_MC = OUTPUT_ROOT / "evaluation" / "multiclass_baseline_resnet18_fullrun"
EVAL_DIR_MC.mkdir(parents=True, exist_ok=True)

METRICS_JSON_MC = EVAL_DIR_MC / "metrics_train_val_test.json"
PREDICTIONS_CSV_MC = EVAL_DIR_MC / "predictions_train_val_test.csv"
CONFUSION_JSON_MC = EVAL_DIR_MC / "confusion_matrices_train_val_test.json"

print("Best checkpoint:", BEST_MODEL_PATH_MC)
print("Eval dir       :", EVAL_DIR_MC)


# Class mapping

IDX_TO_CLASS = {v: k for k, v in MULTICLASS_LABELS.items()}
NUM_CLASSES_MC = len(MULTICLASS_LABELS)


# Recreate model

class ResNet18VideoMulticlassClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_backbone=False):
        super().__init__()

        from torchvision import models
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        video_feats = feats.mean(dim=1)
        logits = self.classifier(video_feats)
        return logits, feats, video_feats


# Helpers

def softmax_np(logits):
    x = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def compute_multiclass_metrics(y_true, logits_np, num_classes):
    y_pred = np.argmax(logits_np, axis=1)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    per_class_accuracy = {}
    for cls_idx in range(num_classes):
        row_sum = cm[cls_idx].sum()
        acc = float(cm[cls_idx, cls_idx] / row_sum) if row_sum > 0 else None
        per_class_accuracy[IDX_TO_CLASS[cls_idx]] = acc

    metrics["per_class_accuracy"] = per_class_accuracy

    return metrics, cm, y_pred

@torch.no_grad()
def evaluate_loader_mc(model, loader, split_name, num_classes, device="cuda"):
    model.eval()

    all_logits = []
    all_labels = []
    all_rows = []

    for xb, yb, metab in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits, frame_feats, video_feats = model(xb)

        logits_np = logits.detach().cpu().numpy()
        labels_np = yb.detach().cpu().numpy()
        probs_np = softmax_np(logits_np)
        preds_np = np.argmax(logits_np, axis=1)

        all_logits.append(logits_np)
        all_labels.append(labels_np)

        for i, meta in enumerate(metab):
            row = {
                "eval_split": split_name,
                "sample_id": meta["sample_id"],
                "dataset": meta["dataset"],
                "family_label_name_true": meta["family_label_name"],
                "multiclass_label_true": int(labels_np[i]),
                "multiclass_label_pred": int(preds_np[i]),
                "family_label_name_pred": IDX_TO_CLASS[int(preds_np[i])],
            }

            for cls_idx in range(num_classes):
                row[f"prob_{IDX_TO_CLASS[cls_idx]}"] = float(probs_np[i, cls_idx])

            all_rows.append(row)

    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)

    metrics, cm, y_pred = compute_multiclass_metrics(labels_np, logits_np, num_classes=num_classes)
    return metrics, cm, pd.DataFrame(all_rows)


# Load checkpoint

checkpoint_mc = torch.load(BEST_MODEL_PATH_MC, map_location=DEVICE)

model_mc = ResNet18VideoMulticlassClassifier(
    num_classes=NUM_CLASSES_MC,
    pretrained=False,
    freeze_backbone=False
).to(DEVICE)

model_mc.load_state_dict(checkpoint_mc["model_state_dict"])
model_mc.eval()

print("Loaded checkpoint epoch:", checkpoint_mc.get("epoch", "unknown"))
print("Best val score         :", checkpoint_mc.get("best_val_score", "unknown"))


# Evaluate all splits

train_metrics_mc, train_cm_mc, train_pred_df_mc = evaluate_loader_mc(
    model_mc, train_loader_mc_full, "train", NUM_CLASSES_MC, device=DEVICE
)
val_metrics_mc, val_cm_mc, val_pred_df_mc = evaluate_loader_mc(
    model_mc, val_loader_mc_full, "val", NUM_CLASSES_MC, device=DEVICE
)
test_metrics_mc, test_cm_mc, test_pred_df_mc = evaluate_loader_mc(
    model_mc, test_loader_mc_full, "test", NUM_CLASSES_MC, device=DEVICE
)

pred_df_mc = pd.concat([train_pred_df_mc, val_pred_df_mc, test_pred_df_mc], axis=0).reset_index(drop=True)
pred_df_mc.to_csv(PREDICTIONS_CSV_MC, index=False)

metrics_summary_mc = {
    "train": train_metrics_mc,
    "val": val_metrics_mc,
    "test": test_metrics_mc,
    "checkpoint_epoch": checkpoint_mc.get("epoch", None),
    "best_val_score": checkpoint_mc.get("best_val_score", None),
}

confusion_summary_mc = {
    "train": train_cm_mc.tolist(),
    "val": val_cm_mc.tolist(),
    "test": test_cm_mc.tolist(),
    "class_order": [IDX_TO_CLASS[i] for i in range(NUM_CLASSES_MC)],
}

with open(METRICS_JSON_MC, "w", encoding="utf-8") as f:
    json.dump(metrics_summary_mc, f, indent=2)

with open(CONFUSION_JSON_MC, "w", encoding="utf-8") as f:
    json.dump(confusion_summary_mc, f, indent=2)


# Print summaries

def print_multiclass_block(name, m):
    print(f"\n{name.upper()} METRICS")
    print(f"  accuracy           : {m['accuracy']:.4f}")
    print(f"  macro_f1           : {m['macro_f1']:.4f}")
    print(f"  weighted_f1        : {m['weighted_f1']:.4f}")
    print(f"  balanced_accuracy  : {m['balanced_accuracy']:.4f}")
    print("  per_class_accuracy :")
    for cls_name, cls_acc in m["per_class_accuracy"].items():
        print(f"    {cls_name:16s}: {cls_acc:.4f}")

print_multiclass_block("train", train_metrics_mc)
print("Train confusion matrix:\n", train_cm_mc)

print_multiclass_block("val", val_metrics_mc)
print("Val confusion matrix:\n", val_cm_mc)

print_multiclass_block("test", test_metrics_mc)
print("Test confusion matrix:\n", test_cm_mc)

print("\nPrediction counts by eval split and predicted family:")
print(pred_df_mc.groupby(["eval_split", "family_label_name_pred"]).size())

print("\nPrediction counts by true/pred family on test:")
print(
    pred_df_mc[pred_df_mc["eval_split"] == "test"]
    .groupby(["family_label_name_true", "family_label_name_pred"])
    .size()
)

print("\nSaved:")
print("  Metrics JSON        :", METRICS_JSON_MC)
print("  Predictions CSV     :", PREDICTIONS_CSV_MC)
print("  Confusion JSON      :", CONFUSION_JSON_MC)

print("\nPredictions preview:")
display(pred_df_mc.head(20))

In [ ]:

# Full-run multiclass forensic report generation


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path


# Inputs

MODEL_READY_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"
MULTICLASS_PREDICTIONS_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "multiclass_baseline_resnet18_fullrun"
    / "predictions_train_val_test.csv"
)


# CHANGE THIS IF NEEDED
# Multiclass uncertainty policy

UNCERTAIN_PROB_THRESHOLD_MC = 0.60
UNCERTAIN_MARGIN_THRESHOLD_MC = 0.15


# Output files

REPORT_DIR_MC = OUTPUT_ROOT / "reports" / "multiclass_baseline_resnet18_fullrun"
REPORT_DIR_MC.mkdir(parents=True, exist_ok=True)

MULTICLASS_REPORT_CSV = REPORT_DIR_MC / "multiclass_forensic_report_table.csv"
MULTICLASS_TEST_PREVIEW_CSV = REPORT_DIR_MC / "multiclass_forensic_report_preview_test.csv"
MULTICLASS_CONFUSION_ANALYSIS_CSV = REPORT_DIR_MC / "multiclass_family_confusion_analysis_test.csv"
MULTICLASS_REPORT_SUMMARY_JSON = REPORT_DIR_MC / "multiclass_forensic_report_summary.json"

print("Model-ready CSV         :", MODEL_READY_CSV)
print("Multiclass predictions  :", MULTICLASS_PREDICTIONS_CSV)
print("Report dir              :", REPORT_DIR_MC)


# Load data

model_ready_df = pd.read_csv(MODEL_READY_CSV, low_memory=False)
pred_df_mc = pd.read_csv(MULTICLASS_PREDICTIONS_CSV)

print("\nLoaded:")
print("  model_ready_df:", model_ready_df.shape)
print("  pred_df_mc    :", pred_df_mc.shape)


# Build one metadata row per sample_id

sample_meta_df = (
    model_ready_df.groupby("sample_id", as_index=False)
    .agg(
        split=("split", "first"),
        dataset=("dataset", "first"),
        source_subset=("source_subset", "first"),
        family_label_name=("family_label_name", "first"),
        binary_label_name=("binary_label_name", "first"),
        relative_path=("relative_path", "first"),
        file_name=("file_name", "first"),
        file_path=("file_path", "first"),
        group_id=("group_id", "first"),
        identity_primary=("identity_primary", "first"),
        identity_secondary=("identity_secondary", "first"),
        n_frames=("frame_index_local", "count"),
        mean_face_confidence=("face_confidence", "mean"),
        mean_face_img_mean=("face_img_mean", "mean"),
        mean_face_img_std=("face_img_std", "mean"),
        any_too_dark=("face_is_too_dark", "max"),
        any_too_bright=("face_is_too_bright", "max"),
        any_low_contrast=("face_is_low_contrast", "max"),
    )
)


# Merge predictions + metadata

report_df_mc = pred_df_mc.merge(
    sample_meta_df,
    on="sample_id",
    how="left",
    suffixes=("_pred", "_meta"),
    validate="one_to_one"
)


# Normalize possibly duplicated column names

def coalesce_columns(df, base_name, pred_name=None, meta_name=None):
    pred_name = pred_name or f"{base_name}_pred"
    meta_name = meta_name or f"{base_name}_meta"

    if pred_name in df.columns and meta_name in df.columns:
        df[base_name] = df[pred_name].combine_first(df[meta_name])
    elif pred_name in df.columns:
        df[base_name] = df[pred_name]
    elif meta_name in df.columns:
        df[base_name] = df[meta_name]
    return df

for col in ["dataset", "family_label_name_true"]:
    report_df_mc = coalesce_columns(report_df_mc, col)


# Probability columns

prob_cols = [c for c in report_df_mc.columns if c.startswith("prob_")]
class_prob_cols = [c for c in prob_cols if c not in ["prob_real", "prob_fake"]]  # safety if old names exist

# We want multiclass class probabilities only
multiclass_prob_cols = []
for cls_name in MULTICLASS_LABELS.keys():
    col = f"prob_{cls_name}"
    if col in report_df_mc.columns:
        multiclass_prob_cols.append(col)

if len(multiclass_prob_cols) != len(MULTICLASS_LABELS):
    raise ValueError(f"Missing multiclass probability columns. Found: {multiclass_prob_cols}")


# Helpers

def provenance_status_from_metadata(row):
    file_path = str(row.get("file_path", ""))
    file_name = str(row.get("file_name", ""))

    if len(file_path) > 0 and len(file_name) > 0:
        return "available"
    return "unavailable"

def confidence_bucket(score):
    if score >= 0.80:
        return "very_high"
    elif score >= 0.60:
        return "high"
    elif score >= 0.40:
        return "moderate"
    else:
        return "low"

def authenticity_from_family(pred_family):
    if pred_family == "real":
        return "real"
    elif pred_family == "uncertain":
        return "uncertain"
    else:
        return "fake"

def explanation_text_mc(row):
    pred_family = row["predicted_family_with_uncertainty"]
    top_prob = float(row["top1_prob"])
    top_margin = float(row["top1_top2_margin"])

    if pred_family == "uncertain":
        return (
            "The multiclass attribution model did not assign a confident forgery-family label. "
            "The sample should be treated as inconclusive and escalated for manual forensic review."
        )

    if pred_family == "real":
        return (
            f"The multiclass attribution model assigns the sample to the real class with confidence "
            f"{top_prob:.3f}. The decision margin is {top_margin:.3f}."
        )

    return (
        f"The multiclass attribution model assigns the sample to the {pred_family} family with confidence "
        f"{top_prob:.3f}. The decision margin over the next candidate class is {top_margin:.3f}."
    )


# Top-1 / top-2 confidence fields

prob_matrix = report_df_mc[multiclass_prob_cols].to_numpy(dtype=float)
top1_idx = np.argmax(prob_matrix, axis=1)
top1_prob = prob_matrix[np.arange(len(report_df_mc)), top1_idx]

sorted_probs = np.sort(prob_matrix, axis=1)
top2_prob = sorted_probs[:, -2]
top1_top2_margin = top1_prob - top2_prob

idx_to_class = {v: k for k, v in MULTICLASS_LABELS.items()}
top1_class = [idx_to_class[i] for i in top1_idx]

report_df_mc["top1_prob"] = top1_prob
report_df_mc["top2_prob"] = top2_prob
report_df_mc["top1_top2_margin"] = top1_top2_margin
report_df_mc["predicted_family_top1"] = top1_class


# Uncertainty-aware predicted family

report_df_mc["predicted_family_with_uncertainty"] = report_df_mc.apply(
    lambda row: (
        row["predicted_family_top1"]
        if (row["top1_prob"] >= UNCERTAIN_PROB_THRESHOLD_MC and row["top1_top2_margin"] >= UNCERTAIN_MARGIN_THRESHOLD_MC)
        else "uncertain"
    ),
    axis=1
)

report_df_mc["predicted_authenticity_from_family"] = report_df_mc["predicted_family_with_uncertainty"].apply(authenticity_from_family)
report_df_mc["confidence_bucket"] = report_df_mc["top1_prob"].apply(confidence_bucket)
report_df_mc["provenance_status"] = report_df_mc.apply(provenance_status_from_metadata, axis=1)
report_df_mc["report_explanation_text"] = report_df_mc.apply(explanation_text_mc, axis=1)

report_df_mc["model_name"] = "multiclass_baseline_resnet18_fullrun"
report_df_mc["uncertain_prob_threshold_mc"] = UNCERTAIN_PROB_THRESHOLD_MC
report_df_mc["uncertain_margin_threshold_mc"] = UNCERTAIN_MARGIN_THRESHOLD_MC


# Test confusion analysis table

test_confusion_df = (
    report_df_mc[report_df_mc["eval_split"] == "test"]
    .groupby(["family_label_name_true", "family_label_name_pred"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

test_true_counts = (
    report_df_mc[report_df_mc["eval_split"] == "test"]
    .groupby("family_label_name_true")
    .size()
    .to_dict()
)

test_confusion_df["proportion_within_true_family"] = test_confusion_df.apply(
    lambda row: row["count"] / max(test_true_counts.get(row["family_label_name_true"], 1), 1),
    axis=1
)

test_confusion_df = test_confusion_df.sort_values(
    by=["family_label_name_true", "count"],
    ascending=[True, False]
).reset_index(drop=True)


# Final column order

final_cols_mc = [
    "eval_split",
    "sample_id",
    "dataset",
    "source_subset",
    "file_name",
    "file_path",
    "relative_path",
    "group_id",
    "identity_primary",
    "identity_secondary",
    "family_label_name_true",
    "family_label_name_pred",
    "predicted_family_top1",
    "predicted_family_with_uncertainty",
    "predicted_authenticity_from_family",
    "multiclass_label_true",
    "multiclass_label_pred",
    "top1_prob",
    "top2_prob",
    "top1_top2_margin",
    "confidence_bucket",
    "provenance_status",
    "n_frames",
    "mean_face_confidence",
    "mean_face_img_mean",
    "mean_face_img_std",
    "any_too_dark",
    "any_too_bright",
    "any_low_contrast",
    "model_name",
    "uncertain_prob_threshold_mc",
    "uncertain_margin_threshold_mc",
    "report_explanation_text",
] + multiclass_prob_cols

final_cols_mc = [c for c in final_cols_mc if c in report_df_mc.columns]
report_df_mc = report_df_mc[final_cols_mc].copy()


# Save files

report_df_mc.to_csv(MULTICLASS_REPORT_CSV, index=False)
report_df_mc[report_df_mc["eval_split"] == "test"].to_csv(MULTICLASS_TEST_PREVIEW_CSV, index=False)
test_confusion_df.to_csv(MULTICLASS_CONFUSION_ANALYSIS_CSV, index=False)


# Summary

summary_mc = {
    "n_total_reports": int(len(report_df_mc)),
    "n_test_reports": int((report_df_mc["eval_split"] == "test").sum()),
    "predicted_family_counts": {
        str(k): int(v)
        for k, v in report_df_mc["family_label_name_pred"].value_counts().to_dict().items()
    },
    "predicted_family_with_uncertainty_counts": {
        str(k): int(v)
        for k, v in report_df_mc["predicted_family_with_uncertainty"].value_counts().to_dict().items()
    },
    "predicted_authenticity_from_family_counts": {
        str(k): int(v)
        for k, v in report_df_mc["predicted_authenticity_from_family"].value_counts().to_dict().items()
    },
    "confidence_bucket_counts": {
        str(k): int(v)
        for k, v in report_df_mc["confidence_bucket"].value_counts().to_dict().items()
    },
}

with open(MULTICLASS_REPORT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_mc, f, indent=2)


# Print summary

print("\nMulticlass forensic report summary:")
print("  Total reports                  :", len(report_df_mc))
print("  Test reports                   :", int((report_df_mc["eval_split"] == "test").sum()))

print("\nPredicted family counts:")
print(report_df_mc["family_label_name_pred"].value_counts())

print("\nPredicted family with uncertainty counts:")
print(report_df_mc["predicted_family_with_uncertainty"].value_counts())

print("\nPredicted authenticity from family counts:")
print(report_df_mc["predicted_authenticity_from_family"].value_counts())

print("\nConfidence bucket counts:")
print(report_df_mc["confidence_bucket"].value_counts())

print("\nTop test-set family confusions:")
display(test_confusion_df.head(20))

print("\nTest report preview:")
display(report_df_mc[report_df_mc["eval_split"] == "test"].head(20))

print("\nSaved:")
print("  Multiclass report CSV          :", MULTICLASS_REPORT_CSV)
print("  Test preview CSV               :", MULTICLASS_TEST_PREVIEW_CSV)
print("  Family confusion analysis CSV  :", MULTICLASS_CONFUSION_ANALYSIS_CSV)
print("  Summary JSON                   :", MULTICLASS_REPORT_SUMMARY_JSON)

In [ ]:

# Full-run multiclass figure generation


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix


# Inputs

METRICS_JSON_MC = (
    OUTPUT_ROOT
    / "evaluation"
    / "multiclass_baseline_resnet18_fullrun"
    / "metrics_train_val_test.json"
)

PREDICTIONS_CSV_MC = (
    OUTPUT_ROOT
    / "evaluation"
    / "multiclass_baseline_resnet18_fullrun"
    / "predictions_train_val_test.csv"
)

CONFUSION_ANALYSIS_CSV_MC = (
    OUTPUT_ROOT
    / "reports"
    / "multiclass_baseline_resnet18_fullrun"
    / "multiclass_family_confusion_analysis_test.csv"
)


# Output figure directory

FIG_DIR_MC = FIGURE_ROOT / "multiclass_baseline_resnet18_fullrun"
FIG_DIR_MC.mkdir(parents=True, exist_ok=True)

FIG_CM_TEST = FIG_DIR_MC / "multiclass_confusion_matrix_test.png"
FIG_PERCLASS_ACC = FIG_DIR_MC / "multiclass_test_per_class_accuracy.png"
FIG_PRED_COUNTS = FIG_DIR_MC / "multiclass_test_predicted_family_counts.png"
FIG_TRUE_PRED_HEATMAP = FIG_DIR_MC / "multiclass_test_true_pred_heatmap.png"
FIG_CONFIDENCE_HIST = FIG_DIR_MC / "multiclass_test_top1_probability_histogram.png"
FIG_MANIFEST_MC = FIG_DIR_MC / "figure_manifest.json"

print("Figure directory:", FIG_DIR_MC)


# Load data

with open(METRICS_JSON_MC, "r", encoding="utf-8") as f:
    metrics_json_mc = json.load(f)

pred_df_mc = pd.read_csv(PREDICTIONS_CSV_MC)
confusion_analysis_df = pd.read_csv(CONFUSION_ANALYSIS_CSV_MC)

print("\nLoaded:")
print("  pred_df_mc            :", pred_df_mc.shape)
print("  confusion_analysis_df :", confusion_analysis_df.shape)


# Class order

class_order = [name for name, idx in sorted(MULTICLASS_LABELS.items(), key=lambda x: x[1])]

test_df_mc = pred_df_mc[pred_df_mc["eval_split"] == "test"].copy().reset_index(drop=True)


# 1) Test confusion matrix heatmap

cm_test = confusion_matrix(
    test_df_mc["family_label_name_true"],
    test_df_mc["family_label_name_pred"],
    labels=class_order
)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_test, interpolation="nearest")
plt.colorbar(im, ax=ax)

ax.set_title("Full-Run Multiclass Test Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_xticks(np.arange(len(class_order)))
ax.set_yticks(np.arange(len(class_order)))
ax.set_xticklabels(class_order, rotation=30, ha="right")
ax.set_yticklabels(class_order)

thresh = cm_test.max() / 2.0 if cm_test.max() > 0 else 0.5
for i in range(cm_test.shape[0]):
    for j in range(cm_test.shape[1]):
        ax.text(
            j, i, format(cm_test[i, j], "d"),
            ha="center", va="center",
            color="white" if cm_test[i, j] > thresh else "black",
            fontsize=9
        )

fig.tight_layout()
fig.savefig(FIG_CM_TEST, dpi=300, bbox_inches="tight")
plt.close(fig)


# 2) Per-class accuracy bar plot

per_class_acc = metrics_json_mc["test"]["per_class_accuracy"]
per_class_df = pd.DataFrame({
    "class_name": list(per_class_acc.keys()),
    "accuracy": list(per_class_acc.values())
}).sort_values("accuracy", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(per_class_df["class_name"], per_class_df["accuracy"])
ax.set_title("Full-Run Multiclass Test Per-Class Accuracy")
ax.set_ylabel("Accuracy")
ax.set_xlabel("Class")
ax.set_ylim(0, 1.0)
ax.set_xticklabels(per_class_df["class_name"], rotation=30, ha="right")
fig.tight_layout()
fig.savefig(FIG_PERCLASS_ACC, dpi=300, bbox_inches="tight")
plt.close(fig)


# 3) Predicted family counts on test

pred_counts_df = (
    test_df_mc["family_label_name_pred"]
    .value_counts()
    .reindex(class_order, fill_value=0)
    .reset_index()
)
pred_counts_df.columns = ["family", "count"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(pred_counts_df["family"], pred_counts_df["count"])
ax.set_title("Full-Run Multiclass Test Predicted Family Counts")
ax.set_ylabel("Count")
ax.set_xlabel("Predicted family")
ax.set_xticklabels(pred_counts_df["family"], rotation=30, ha="right")
fig.tight_layout()
fig.savefig(FIG_PRED_COUNTS, dpi=300, bbox_inches="tight")
plt.close(fig)


# 4) Normalized true-vs-pred heatmap

heatmap_df = confusion_analysis_df.pivot(
    index="family_label_name_true",
    columns="family_label_name_pred",
    values="proportion_within_true_family"
).fillna(0.0)

heatmap_df = heatmap_df.reindex(index=class_order, columns=class_order, fill_value=0.0)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(heatmap_df.values, interpolation="nearest", vmin=0.0, vmax=1.0)
plt.colorbar(im, ax=ax)

ax.set_title("Normalized True-to-Predicted Family Confusion (Test)")
ax.set_xlabel("Predicted family")
ax.set_ylabel("True family")
ax.set_xticks(np.arange(len(class_order)))
ax.set_yticks(np.arange(len(class_order)))
ax.set_xticklabels(class_order, rotation=30, ha="right")
ax.set_yticklabels(class_order)

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        val = heatmap_df.values[i, j]
        ax.text(
            j, i, f"{val:.2f}",
            ha="center", va="center",
            color="white" if val > 0.5 else "black",
            fontsize=8
        )

fig.tight_layout()
fig.savefig(FIG_TRUE_PRED_HEATMAP, dpi=300, bbox_inches="tight")
plt.close(fig)


# 5) Top-1 confidence histogram on test

prob_cols_mc = [f"prob_{cls}" for cls in class_order]
top1_prob_test = test_df_mc[prob_cols_mc].max(axis=1)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.hist(top1_prob_test, bins=25)
ax.axvline(0.60, linestyle="--", label="Uncertainty prob threshold = 0.60")
ax.set_title("Full-Run Multiclass Test Top-1 Probability Distribution")
ax.set_xlabel("Top-1 predicted probability")
ax.set_ylabel("Number of samples")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIG_CONFIDENCE_HIST, dpi=300, bbox_inches="tight")
plt.close(fig)


# Save manifest

manifest_mc = {
    "confusion_matrix_test": str(FIG_CM_TEST),
    "per_class_accuracy_plot": str(FIG_PERCLASS_ACC),
    "predicted_family_counts_plot": str(FIG_PRED_COUNTS),
    "normalized_true_pred_heatmap": str(FIG_TRUE_PRED_HEATMAP),
    "top1_probability_histogram": str(FIG_CONFIDENCE_HIST),
    "class_order": class_order,
}

with open(FIG_MANIFEST_MC, "w", encoding="utf-8") as f:
    json.dump(manifest_mc, f, indent=2)

print("\nSaved figures:")
print("  Test confusion matrix   :", FIG_CM_TEST)
print("  Per-class accuracy      :", FIG_PERCLASS_ACC)
print("  Predicted family counts :", FIG_PRED_COUNTS)
print("  True-pred heatmap       :", FIG_TRUE_PRED_HEATMAP)
print("  Top-1 confidence hist   :", FIG_CONFIDENCE_HIST)
print("  Figure manifest         :", FIG_MANIFEST_MC)

In [ ]:

# Representative multiclass test-sample selection


import os
import json
import numpy as np
import pandas as pd
from pathlib import Path


# Inputs

MODEL_READY_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"
MULTICLASS_PREDICTIONS_CSV = (
    OUTPUT_ROOT
    / "evaluation"
    / "multiclass_baseline_resnet18_fullrun"
    / "predictions_train_val_test.csv"
)


# CHANGE THIS IF NEEDED
# Selection policy

N_CORRECT_PER_CLASS = 2
N_HARD_PER_CLASS = 1

# If True, hard examples are chosen from:
#   - misclassified first
#   - otherwise lowest-margin correct examples
PREFER_MISCLASSIFIED_FOR_HARD = True


# Output files

EXPLAIN_DIR_MC = OUTPUT_ROOT / "explainability" / "multiclass_baseline_resnet18_gradcam"
EXPLAIN_DIR_MC.mkdir(parents=True, exist_ok=True)

SELECTED_SAMPLES_CSV = EXPLAIN_DIR_MC / "selected_test_samples_for_gradcam.csv"
SELECTION_SUMMARY_JSON = EXPLAIN_DIR_MC / "selected_test_samples_summary.json"

print("Model-ready CSV        :", MODEL_READY_CSV)
print("Multiclass predictions :", MULTICLASS_PREDICTIONS_CSV)
print("Explainability dir     :", EXPLAIN_DIR_MC)


# Load data

model_ready_df = pd.read_csv(MODEL_READY_CSV, low_memory=False)
pred_df_mc = pd.read_csv(MULTICLASS_PREDICTIONS_CSV)

print("\nLoaded:")
print("  model_ready_df:", model_ready_df.shape)
print("  pred_df_mc    :", pred_df_mc.shape)


# Build one metadata row per sample_id

sample_meta_df = (
    model_ready_df.groupby("sample_id", as_index=False)
    .agg(
        split=("split", "first"),
        dataset=("dataset", "first"),
        source_subset=("source_subset", "first"),
        family_label_name=("family_label_name", "first"),
        binary_label_name=("binary_label_name", "first"),
        relative_path=("relative_path", "first"),
        file_name=("file_name", "first"),
        file_path=("file_path", "first"),
        group_id=("group_id", "first"),
        identity_primary=("identity_primary", "first"),
        identity_secondary=("identity_secondary", "first"),
        n_frames=("frame_index_local", "count"),
        mean_face_confidence=("face_confidence", "mean"),
        mean_face_img_mean=("face_img_mean", "mean"),
        mean_face_img_std=("face_img_std", "mean"),
    )
)


# Merge predictions + metadata

df = pred_df_mc.merge(
    sample_meta_df,
    on="sample_id",
    how="left",
    suffixes=("_pred", "_meta"),
    validate="one_to_one"
)

# normalize duplicated names if present
if "dataset_pred" in df.columns and "dataset_meta" in df.columns:
    df["dataset"] = df["dataset_pred"].combine_first(df["dataset_meta"])
elif "dataset_pred" in df.columns:
    df["dataset"] = df["dataset_pred"]
elif "dataset_meta" in df.columns:
    df["dataset"] = df["dataset_meta"]

if "family_label_name_true" not in df.columns and "family_label_name" in df.columns:
    df["family_label_name_true"] = df["family_label_name"]


# Restrict to test split

test_df = df[df["eval_split"] == "test"].copy().reset_index(drop=True)


# Compute top1/top2 stats

class_order = [name for name, idx in sorted(MULTICLASS_LABELS.items(), key=lambda x: x[1])]
prob_cols = [f"prob_{cls}" for cls in class_order]

missing_prob_cols = [c for c in prob_cols if c not in test_df.columns]
if missing_prob_cols:
    raise ValueError(f"Missing probability columns: {missing_prob_cols}")

prob_matrix = test_df[prob_cols].to_numpy(dtype=float)
top1_idx = np.argmax(prob_matrix, axis=1)
top1_prob = prob_matrix[np.arange(len(test_df)), top1_idx]
sorted_probs = np.sort(prob_matrix, axis=1)
top2_prob = sorted_probs[:, -2]
top1_top2_margin = top1_prob - top2_prob

idx_to_class = {v: k for k, v in MULTICLASS_LABELS.items()}
top1_class = [idx_to_class[i] for i in top1_idx]

test_df["predicted_top1_class"] = top1_class
test_df["top1_prob"] = top1_prob
test_df["top2_prob"] = top2_prob
test_df["top1_top2_margin"] = top1_top2_margin
test_df["is_correct"] = (
    test_df["family_label_name_true"] == test_df["family_label_name_pred"]
).astype(int)


# Selection logic

selected_parts = []

for cls_name in class_order:
    cls_df = test_df[test_df["family_label_name_true"] == cls_name].copy()

    # 1) correct high-confidence examples
    correct_df = cls_df[cls_df["is_correct"] == 1].copy()
    correct_df = correct_df.sort_values(
        by=["top1_prob", "top1_top2_margin"],
        ascending=[False, False]
    ).head(N_CORRECT_PER_CLASS).copy()
    correct_df["selection_type"] = "correct_high_confidence"
    selected_parts.append(correct_df)

    # 2) hard examples
    if PREFER_MISCLASSIFIED_FOR_HARD:
        hard_df = cls_df[cls_df["is_correct"] == 0].copy()
        hard_df = hard_df.sort_values(
            by=["top1_prob", "top1_top2_margin"],
            ascending=[False, False]
        ).head(N_HARD_PER_CLASS).copy()

        if len(hard_df) < N_HARD_PER_CLASS:
            needed = N_HARD_PER_CLASS - len(hard_df)
            low_margin_correct = cls_df[cls_df["is_correct"] == 1].copy()
            low_margin_correct = low_margin_correct.sort_values(
                by=["top1_top2_margin", "top1_prob"],
                ascending=[True, True]
            ).head(needed).copy()
            hard_df = pd.concat([hard_df, low_margin_correct], axis=0)
    else:
        hard_df = cls_df.sort_values(
            by=["top1_top2_margin", "top1_prob"],
            ascending=[True, True]
        ).head(N_HARD_PER_CLASS).copy()

    if len(hard_df) > 0:
        hard_df["selection_type"] = "hard_or_ambiguous"
        selected_parts.append(hard_df)

selected_df = pd.concat(selected_parts, axis=0).drop_duplicates(subset=["sample_id"]).reset_index(drop=True)


# Add concise ranking info

selected_df["selection_rank_within_true_class"] = (
    selected_df.groupby(["family_label_name_true", "selection_type"])
    .cumcount() + 1
)

selected_df = selected_df.sort_values(
    by=["family_label_name_true", "selection_type", "selection_rank_within_true_class"]
).reset_index(drop=True)


# Save final selection table

selection_cols = [
    "sample_id",
    "eval_split",
    "dataset",
    "source_subset",
    "file_name",
    "file_path",
    "relative_path",
    "group_id",
    "identity_primary",
    "identity_secondary",
    "family_label_name_true",
    "family_label_name_pred",
    "multiclass_label_true",
    "multiclass_label_pred",
    "is_correct",
    "selection_type",
    "selection_rank_within_true_class",
    "top1_prob",
    "top2_prob",
    "top1_top2_margin",
] + prob_cols

selection_cols = [c for c in selection_cols if c in selected_df.columns]
selected_df = selected_df[selection_cols].copy()

selected_df.to_csv(SELECTED_SAMPLES_CSV, index=False)


# Summary

summary = {
    "n_selected_samples": int(len(selected_df)),
    "n_classes": int(len(class_order)),
    "n_correct_per_class_target": int(N_CORRECT_PER_CLASS),
    "n_hard_per_class_target": int(N_HARD_PER_CLASS),
    "selected_counts_by_true_class": {
        str(k): int(v)
        for k, v in selected_df["family_label_name_true"].value_counts().sort_index().to_dict().items()
    },
    "selected_counts_by_selection_type": {
        str(k): int(v)
        for k, v in selected_df["selection_type"].value_counts().to_dict().items()
    },
    "correctness_counts": {
        str(k): int(v)
        for k, v in selected_df["is_correct"].value_counts().to_dict().items()
    },
}

with open(SELECTION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# Print summary

print("\nSelection summary:")
print("  Selected samples total :", len(selected_df))

print("\nSelected counts by true class:")
print(selected_df["family_label_name_true"].value_counts().sort_index())

print("\nSelected counts by selection type:")
print(selected_df["selection_type"].value_counts())

print("\nSelected correctness counts:")
print(selected_df["is_correct"].value_counts())

print("\nSelected sample preview:")
display(selected_df.head(20))

print("\nSaved:")
print("  Selected samples CSV   :", SELECTED_SAMPLES_CSV)
print("  Selection summary JSON :", SELECTION_SUMMARY_JSON)

In [ ]:

# (CORRECTED): Full-run multiclass Grad-CAM generation


import os
import json
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms


# Inputs

SELECTED_SAMPLES_CSV = (
    OUTPUT_ROOT
    / "explainability"
    / "multiclass_baseline_resnet18_gradcam"
    / "selected_test_samples_for_gradcam.csv"
)

MODEL_READY_CSV = METADATA_ROOT / "metadata_selected_videos_fullrun_model_ready.csv"
BEST_MODEL_PATH_MC = CHECKPOINT_ROOT / "multiclass_baseline_resnet18_fullrun" / "best_model.pt"


# Output files

EXPLAIN_DIR_MC = OUTPUT_ROOT / "explainability" / "multiclass_baseline_resnet18_gradcam"
EXPLAIN_DIR_MC.mkdir(parents=True, exist_ok=True)

GRADCAM_MANIFEST_CSV_MC = EXPLAIN_DIR_MC / "gradcam_manifest.csv"
GRADCAM_SUMMARY_JSON_MC = EXPLAIN_DIR_MC / "gradcam_summary.json"

print("Selected samples CSV :", SELECTED_SAMPLES_CSV)
print("Model-ready CSV      :", MODEL_READY_CSV)
print("Best checkpoint      :", BEST_MODEL_PATH_MC)
print("Explainability dir   :", EXPLAIN_DIR_MC)


# CHANGE THIS IF NEEDED
# Device / settings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = FACE_IMAGE_SIZE
ALPHA_OVERLAY = 0.40

print("\nUsing device:", DEVICE)


# Load data

selected_df = pd.read_csv(SELECTED_SAMPLES_CSV)
model_ready_df = pd.read_csv(MODEL_READY_CSV, low_memory=False)

print("Selected samples:", selected_df.shape)
print("Model-ready rows :", model_ready_df.shape)

selected_ids = selected_df["sample_id"].tolist()

subset_df = model_ready_df[model_ready_df["sample_id"].isin(selected_ids)].copy()
subset_df = subset_df.sort_values(["sample_id", "frame_index_local"]).reset_index(drop=True)

print("Subset frame rows for explanation:", subset_df.shape)


# Model definition

NUM_CLASSES_MC = len(MULTICLASS_LABELS)
IDX_TO_CLASS = {v: k for k, v in MULTICLASS_LABELS.items()}

class ResNet18VideoMulticlassClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True, freeze_backbone=False):
        super().__init__()

        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        video_feats = feats.mean(dim=1)
        logits = self.classifier(video_feats)
        return logits, feats, video_feats


# Load checkpoint

checkpoint_mc = torch.load(BEST_MODEL_PATH_MC, map_location=DEVICE)

model_mc = ResNet18VideoMulticlassClassifier(
    num_classes=NUM_CLASSES_MC,
    pretrained=False,
    freeze_backbone=False
).to(DEVICE)

model_mc.load_state_dict(checkpoint_mc["model_state_dict"])
model_mc.eval()

print("Loaded checkpoint epoch:", checkpoint_mc.get("epoch", "unknown"))


# Grad-CAM hooks

target_layer = model_mc.backbone.layer4[-1].conv2

activations = []
gradients = []

def forward_hook(module, inp, out):
    activations.append(out.detach())

def backward_hook(module, grad_input, grad_output):
    gradients.append(grad_output[0].detach())

fh = target_layer.register_forward_hook(forward_hook)
bh = target_layer.register_full_backward_hook(backward_hook)


# Preprocess helpers

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def load_frame_tensor(image_path):
    img = Image.open(image_path).convert("RGB")
    x = eval_transform(img)
    return x

def tensor_to_rgb_uint8(image_path):
    img = Image.open(image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.array(img, dtype=np.uint8)

def make_gradcam_from_last_activation_and_gradient(act, grad, out_size=(224, 224)):
    # act: [C, H, W]
    # grad: [C, H, W]
    weights = grad.mean(dim=(1, 2), keepdim=True)   # [C,1,1]
    cam = (weights * act).sum(dim=0)                # [H,W]
    cam = torch.relu(cam)

    cam = cam.cpu().numpy()

    if cam.max() > 0:
        cam = cam / cam.max()

    # Resize from feature-map resolution (e.g. 7x7) to image size (224x224)
    cam_resized = cv2.resize(cam, out_size, interpolation=cv2.INTER_CUBIC)

    # Normalize again after resize
    cam_resized = np.maximum(cam_resized, 0)
    if cam_resized.max() > 0:
        cam_resized = cam_resized / cam_resized.max()

    return cam_resized

def overlay_heatmap_on_rgb(rgb_img, cam, alpha=0.4):
    # rgb_img: [H,W,3], cam: [H,W]
    heatmap = np.uint8(255 * cam)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = ((1 - alpha) * rgb_img.astype(np.float32) + alpha * heatmap.astype(np.float32))
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    return heatmap, overlay


# Run explainability

records = []
n_samples_done = 0
n_frames_done = 0

for sample_id in selected_ids:
    sample_rows = subset_df[subset_df["sample_id"] == sample_id].copy().sort_values("frame_index_local")
    if len(sample_rows) == 0:
        continue

    sample_meta = selected_df[selected_df["sample_id"] == sample_id].iloc[0]

    frame_paths = sample_rows["face_crop_path"].tolist()
    frame_indices = sample_rows["frame_index_local"].tolist()

    if len(frame_paths) != 8:
        print(f"[WARN] Skipping {sample_id}: expected 8 frames, found {len(frame_paths)}")
        continue

    x_list = [load_frame_tensor(p) for p in frame_paths]
    x = torch.stack(x_list, dim=0).unsqueeze(0).to(DEVICE)  # [1,T,C,H,W]

    activations.clear()
    gradients.clear()

    logits, _, _ = model_mc(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).item())
    pred_prob = float(probs[pred_idx].item())
    pred_class = IDX_TO_CLASS[pred_idx]

    model_mc.zero_grad(set_to_none=True)
    target_score = logits[0, pred_idx]
    target_score.backward()

    act_bt = activations[-1]   # [B*T, C, H, W]
    grad_bt = gradients[-1]    # [B*T, C, H, W]

    sample_dir = EXPLAIN_DIR_MC / f"{sample_meta['family_label_name_true']}__{sample_id}"
    raw_dir = sample_dir / "raw"
    heatmap_dir = sample_dir / "heatmap"
    overlay_dir = sample_dir / "overlay"

    raw_dir.mkdir(parents=True, exist_ok=True)
    heatmap_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    for i, (frame_idx_local, frame_path) in enumerate(zip(frame_indices, frame_paths)):
        rgb_img = tensor_to_rgb_uint8(frame_path)
        cam = make_gradcam_from_last_activation_and_gradient(
            act_bt[i], grad_bt[i], out_size=(rgb_img.shape[1], rgb_img.shape[0])
        )
        heatmap_rgb, overlay_rgb = overlay_heatmap_on_rgb(rgb_img, cam, alpha=ALPHA_OVERLAY)

        raw_out = raw_dir / f"frame_{int(frame_idx_local):02d}.png"
        heatmap_out = heatmap_dir / f"heatmap_{int(frame_idx_local):02d}.png"
        overlay_out = overlay_dir / f"overlay_{int(frame_idx_local):02d}.png"

        Image.fromarray(rgb_img).save(raw_out)
        Image.fromarray(heatmap_rgb).save(heatmap_out)
        Image.fromarray(overlay_rgb).save(overlay_out)

        records.append({
            "sample_id": sample_id,
            "dataset": sample_meta["dataset"],
            "selection_type": sample_meta["selection_type"],
            "true_family_label": sample_meta["family_label_name_true"],
            "predicted_family_label": sample_meta["family_label_name_pred"],
            "is_correct": int(sample_meta["is_correct"]),
            "top1_prob": float(sample_meta["top1_prob"]),
            "top2_prob": float(sample_meta["top2_prob"]),
            "top1_top2_margin": float(sample_meta["top1_top2_margin"]),
            "gradcam_target_class": pred_class,
            "gradcam_target_prob": pred_prob,
            "frame_index_local": int(frame_idx_local),
            "face_crop_path": str(frame_path),
            "raw_image_path": str(raw_out),
            "heatmap_path": str(heatmap_out),
            "overlay_path": str(overlay_out),
        })

        n_frames_done += 1

    n_samples_done += 1
    print(f"Explained {n_samples_done}/{len(selected_ids)} samples")


# Save manifest and summary

gradcam_df_mc = pd.DataFrame(records)
gradcam_df_mc.to_csv(GRADCAM_MANIFEST_CSV_MC, index=False)

summary_mc = {
    "n_selected_samples": int(len(selected_ids)),
    "n_samples_explained": int(n_samples_done),
    "n_frames_explained": int(n_frames_done),
    "checkpoint_epoch": checkpoint_mc.get("epoch", None),
    "class_counts_true": {
        str(k): int(v)
        for k, v in gradcam_df_mc["true_family_label"].value_counts().sort_index().to_dict().items()
    } if len(gradcam_df_mc) > 0 else {},
    "selection_type_counts": {
        str(k): int(v)
        for k, v in gradcam_df_mc["selection_type"].value_counts().to_dict().items()
    } if len(gradcam_df_mc) > 0 else {},
}

with open(GRADCAM_SUMMARY_JSON_MC, "w", encoding="utf-8") as f:
    json.dump(summary_mc, f, indent=2)

fh.remove()
bh.remove()


# Print summary

print("\nMulticlass Grad-CAM summary:")
print("  Samples explained   :", n_samples_done)
print("  Frames explained    :", n_frames_done)

print("\nSaved:")
print("  Grad-CAM manifest   :", GRADCAM_MANIFEST_CSV_MC)
print("  Grad-CAM summary    :", GRADCAM_SUMMARY_JSON_MC)

print("\nPreview:")
display(gradcam_df_mc.head(20))

In [ ]:

# Multiclass qualitative panel generation


import os
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt


# Inputs

SELECTED_SAMPLES_CSV = (
    OUTPUT_ROOT
    / "explainability"
    / "multiclass_baseline_resnet18_gradcam"
    / "selected_test_samples_for_gradcam.csv"
)

GRADCAM_MANIFEST_CSV_MC = (
    OUTPUT_ROOT
    / "explainability"
    / "multiclass_baseline_resnet18_gradcam"
    / "gradcam_manifest.csv"
)


# CHANGE THIS IF NEEDED
# Panel configuration

TRUE_CLASS_ORDER = [name for name, idx in sorted(MULTICLASS_LABELS.items(), key=lambda x: x[1])]

# Use these frame indices from each 8-frame clip for compact figure display
PANEL_FRAME_INDICES = [0, 3, 7]

# If True, pick correct high-confidence first; fallback to hard sample if needed
PREFER_CORRECT_SAMPLE = True


# Output files

PANEL_DIR = OUTPUT_ROOT / "figures" / "multiclass_baseline_resnet18_gradcam_panels"
PANEL_DIR.mkdir(parents=True, exist_ok=True)

QUAL_PANEL_PNG = PANEL_DIR / "multiclass_gradcam_qualitative_panel.png"
PANEL_MANIFEST_JSON = PANEL_DIR / "multiclass_gradcam_qualitative_panel_manifest.json"

print("Selected samples CSV :", SELECTED_SAMPLES_CSV)
print("Grad-CAM manifest    :", GRADCAM_MANIFEST_CSV_MC)
print("Panel dir            :", PANEL_DIR)


# Load data

selected_df = pd.read_csv(SELECTED_SAMPLES_CSV)
gradcam_df = pd.read_csv(GRADCAM_MANIFEST_CSV_MC)

print("\nLoaded:")
print("  selected_df :", selected_df.shape)
print("  gradcam_df  :", gradcam_df.shape)


# Choose one representative sample per true class

chosen_rows = []

for cls_name in TRUE_CLASS_ORDER:
    cls_df = selected_df[selected_df["family_label_name_true"] == cls_name].copy()

    chosen = None

    if PREFER_CORRECT_SAMPLE:
        correct_df = cls_df[cls_df["selection_type"] == "correct_high_confidence"].copy()
        if len(correct_df) > 0:
            correct_df = correct_df.sort_values(
                by=["top1_prob", "top1_top2_margin"],
                ascending=[False, False]
            )
            chosen = correct_df.iloc[0]

    if chosen is None:
        cls_df = cls_df.sort_values(
            by=["is_correct", "top1_prob", "top1_top2_margin"],
            ascending=[False, False, False]
        )
        if len(cls_df) > 0:
            chosen = cls_df.iloc[0]

    if chosen is not None:
        chosen_rows.append(chosen)

chosen_df = pd.DataFrame(chosen_rows).reset_index(drop=True)

print("\nChosen samples for panel:")
print(chosen_df[[
    "sample_id", "family_label_name_true", "family_label_name_pred",
    "selection_type", "is_correct", "top1_prob", "top1_top2_margin"
]])


# Build figure grid
# Rows = classes
# Cols = selected frames

n_rows = len(chosen_df)
n_cols = len(PANEL_FRAME_INDICES)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.2 * n_cols, 2.8 * n_rows),
    squeeze=False
)

panel_records = []

for row_idx, chosen in chosen_df.iterrows():
    sample_id = chosen["sample_id"]
    true_cls = chosen["family_label_name_true"]
    pred_cls = chosen["family_label_name_pred"]
    selection_type = chosen["selection_type"]
    is_correct = int(chosen["is_correct"])
    top1_prob = float(chosen["top1_prob"])
    margin = float(chosen["top1_top2_margin"])

    sample_gradcam = gradcam_df[gradcam_df["sample_id"] == sample_id].copy()
    sample_gradcam = sample_gradcam.sort_values("frame_index_local")

    for col_idx, frame_idx in enumerate(PANEL_FRAME_INDICES):
        ax = axes[row_idx, col_idx]

        row_match = sample_gradcam[sample_gradcam["frame_index_local"] == frame_idx].copy()

        if len(row_match) == 0:
            ax.axis("off")
            continue

        row_match = row_match.iloc[0]
        overlay_path = row_match["overlay_path"]

        img = Image.open(overlay_path).convert("RGB")
        ax.imshow(img)
        ax.axis("off")

        if row_idx == 0:
            ax.set_title(f"Frame {frame_idx}", fontsize=11)

        if col_idx == 0:
            left_title = (
                f"True: {true_cls}\n"
                f"Pred: {pred_cls}\n"
                f"{'Correct' if is_correct else 'Misclassified'} | "
                f"{selection_type}\n"
                f"p={top1_prob:.3f}, m={margin:.3f}"
            )
            ax.text(
                -0.08, 0.5, left_title,
                transform=ax.transAxes,
                va="center", ha="right",
                fontsize=10
            )

        panel_records.append({
            "sample_id": sample_id,
            "true_class": true_cls,
            "pred_class": pred_cls,
            "selection_type": selection_type,
            "is_correct": is_correct,
            "top1_prob": top1_prob,
            "top1_top2_margin": margin,
            "frame_index_local": int(frame_idx),
            "overlay_path": str(overlay_path),
        })

fig.suptitle(
    "Representative Multiclass Grad-CAM Overlays Across Forgery Families",
    fontsize=16, y=0.995
)

fig.tight_layout(rect=[0.10, 0.02, 1.0, 0.985])
fig.savefig(QUAL_PANEL_PNG, dpi=300, bbox_inches="tight")
plt.close(fig)


# Save manifest

manifest = {
    "panel_path": str(QUAL_PANEL_PNG),
    "n_rows": int(n_rows),
    "n_cols": int(n_cols),
    "panel_frame_indices": [int(x) for x in PANEL_FRAME_INDICES],
    "selected_samples": panel_records,
}

with open(PANEL_MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved:")
print("  Qualitative panel PNG  :", QUAL_PANEL_PNG)
print("  Panel manifest JSON    :", PANEL_MANIFEST_JSON)

In [ ]:

# Hard-case multiclass Grad-CAM panel

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt


# Inputs

SELECTED_SAMPLES_CSV = (
    OUTPUT_ROOT
    / "explainability"
    / "multiclass_baseline_resnet18_gradcam"
    / "selected_test_samples_for_gradcam.csv"
)

GRADCAM_MANIFEST_CSV_MC = (
    OUTPUT_ROOT
    / "explainability"
    / "multiclass_baseline_resnet18_gradcam"
    / "gradcam_manifest.csv"
)


# CHANGE THIS IF NEEDED
# Panel configuration

PANEL_FRAME_INDICES = [0, 3, 7]

# maximum hard cases to show
MAX_HARD_CASES = 7

# sort hard cases by highest confidence first
SORT_HARD_BY = ["top1_prob", "top1_top2_margin"]
SORT_ASC = [False, False]


# Output files

PANEL_DIR = OUTPUT_ROOT / "figures" / "multiclass_baseline_resnet18_gradcam_panels"
PANEL_DIR.mkdir(parents=True, exist_ok=True)

HARD_PANEL_PNG = PANEL_DIR / "multiclass_gradcam_hardcase_panel.png"
HARD_PANEL_MANIFEST_JSON = PANEL_DIR / "multiclass_gradcam_hardcase_panel_manifest.json"

print("Selected samples CSV :", SELECTED_SAMPLES_CSV)
print("Grad-CAM manifest    :", GRADCAM_MANIFEST_CSV_MC)
print("Panel dir            :", PANEL_DIR)


# Load data

selected_df = pd.read_csv(SELECTED_SAMPLES_CSV)
gradcam_df = pd.read_csv(GRADCAM_MANIFEST_CSV_MC)

print("\nLoaded:")
print("  selected_df :", selected_df.shape)
print("  gradcam_df  :", gradcam_df.shape)


# Choose hard / ambiguous samples

hard_df = selected_df[selected_df["selection_type"] == "hard_or_ambiguous"].copy()

if len(hard_df) == 0:
    raise RuntimeError("No hard_or_ambiguous samples found in selected_df.")

hard_df = hard_df.sort_values(by=SORT_HARD_BY, ascending=SORT_ASC).head(MAX_HARD_CASES).reset_index(drop=True)

print("\nChosen hard cases:")
print(hard_df[[
    "sample_id",
    "family_label_name_true",
    "family_label_name_pred",
    "is_correct",
    "top1_prob",
    "top1_top2_margin"
]])


# Build figure grid

n_rows = len(hard_df)
n_cols = len(PANEL_FRAME_INDICES)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.2 * n_cols, 2.8 * n_rows),
    squeeze=False
)

panel_records = []

for row_idx, chosen in hard_df.iterrows():
    sample_id = chosen["sample_id"]
    true_cls = chosen["family_label_name_true"]
    pred_cls = chosen["family_label_name_pred"]
    is_correct = int(chosen["is_correct"])
    top1_prob = float(chosen["top1_prob"])
    margin = float(chosen["top1_top2_margin"])

    sample_gradcam = gradcam_df[gradcam_df["sample_id"] == sample_id].copy()
    sample_gradcam = sample_gradcam.sort_values("frame_index_local")

    for col_idx, frame_idx in enumerate(PANEL_FRAME_INDICES):
        ax = axes[row_idx, col_idx]

        row_match = sample_gradcam[sample_gradcam["frame_index_local"] == frame_idx].copy()

        if len(row_match) == 0:
            ax.axis("off")
            continue

        row_match = row_match.iloc[0]
        overlay_path = row_match["overlay_path"]

        img = Image.open(overlay_path).convert("RGB")
        ax.imshow(img)
        ax.axis("off")

        if row_idx == 0:
            ax.set_title(f"Frame {frame_idx}", fontsize=11)

        if col_idx == 0:
            left_title = (
                f"True: {true_cls}\n"
                f"Pred: {pred_cls}\n"
                f"{'Correct but hard' if is_correct else 'Misclassified'}\n"
                f"p={top1_prob:.3f}, m={margin:.3f}"
            )
            ax.text(
                -0.08, 0.5, left_title,
                transform=ax.transAxes,
                va="center", ha="right",
                fontsize=10
            )

        panel_records.append({
            "sample_id": sample_id,
            "true_class": true_cls,
            "pred_class": pred_cls,
            "is_correct": is_correct,
            "top1_prob": top1_prob,
            "top1_top2_margin": margin,
            "frame_index_local": int(frame_idx),
            "overlay_path": str(overlay_path),
        })

fig.suptitle(
    "Hard-Case Multiclass Grad-CAM Overlays for Forensic Error Analysis",
    fontsize=16, y=0.995
)

fig.tight_layout(rect=[0.10, 0.02, 1.0, 0.985])
fig.savefig(HARD_PANEL_PNG, dpi=300, bbox_inches="tight")
plt.close(fig)


# Save manifest

manifest = {
    "panel_path": str(HARD_PANEL_PNG),
    "n_rows": int(n_rows),
    "n_cols": int(n_cols),
    "panel_frame_indices": [int(x) for x in PANEL_FRAME_INDICES],
    "selected_samples": panel_records,
}

with open(HARD_PANEL_MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved:")
print("  Hard-case panel PNG   :", HARD_PANEL_PNG)
print("  Panel manifest JSON   :", HARD_PANEL_MANIFEST_JSON)

In [ ]:

# Multiclass temporal explanation consistency analysis

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn.metrics.pairwise import cosine_similarity


# Inputs

GRADCAM_MANIFEST_CSV_MC = (
    OUTPUT_ROOT
    / "explainability"
    / "multiclass_baseline_resnet18_gradcam"
    / "gradcam_manifest.csv"
)


# Output files

TEMPORAL_DIR_MC = OUTPUT_ROOT / "explainability" / "multiclass_baseline_resnet18_temporal_consistency"
TEMPORAL_DIR_MC.mkdir(parents=True, exist_ok=True)

FRAMEPAIR_CSV_MC = TEMPORAL_DIR_MC / "framepair_temporal_consistency.csv"
VIDEO_SUMMARY_CSV_MC = TEMPORAL_DIR_MC / "video_temporal_consistency_summary.csv"
TEMPORAL_SUMMARY_JSON_MC = TEMPORAL_DIR_MC / "temporal_consistency_summary.json"

print("Grad-CAM manifest :", GRADCAM_MANIFEST_CSV_MC)
print("Temporal dir      :", TEMPORAL_DIR_MC)


# Load manifest

gradcam_df = pd.read_csv(GRADCAM_MANIFEST_CSV_MC)

print("\nLoaded manifest:", gradcam_df.shape)

required_cols = [
    "sample_id",
    "true_family_label",
    "predicted_family_label",
    "is_correct",
    "selection_type",
    "frame_index_local",
    "heatmap_path",
]
missing_cols = [c for c in required_cols if c not in gradcam_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")


# Helpers

def load_heatmap_as_gray_float(path):
    img = Image.open(path).convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr

def safe_pearson(a, b):
    a = a.flatten()
    b = b.flatten()
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def safe_cosine(a, b):
    a = a.flatten().reshape(1, -1)
    b = b.flatten().reshape(1, -1)
    denom_a = np.linalg.norm(a)
    denom_b = np.linalg.norm(b)
    if denom_a < 1e-8 or denom_b < 1e-8:
        return np.nan
    return float(cosine_similarity(a, b)[0, 0])

def l1_similarity(a, b):
    # convert L1 distance into a similarity-like score in [0,1]
    dist = np.mean(np.abs(a.flatten() - b.flatten()))
    sim = 1.0 - dist
    return float(sim)


# Frame-pair temporal analysis

framepair_records = []
video_summary_records = []

for sample_id, sub_df in gradcam_df.groupby("sample_id"):
    sub_df = sub_df.sort_values("frame_index_local").reset_index(drop=True)

    if len(sub_df) < 2:
        continue

    pearsons = []
    cosines = []
    l1s = []

    for i in range(len(sub_df) - 1):
        row_a = sub_df.iloc[i]
        row_b = sub_df.iloc[i + 1]

        heat_a = load_heatmap_as_gray_float(row_a["heatmap_path"])
        heat_b = load_heatmap_as_gray_float(row_b["heatmap_path"])

        pearson_val = safe_pearson(heat_a, heat_b)
        cosine_val = safe_cosine(heat_a, heat_b)
        l1_val = l1_similarity(heat_a, heat_b)

        framepair_records.append({
            "sample_id": sample_id,
            "dataset": row_a.get("dataset", None),
            "true_family_label": row_a["true_family_label"],
            "predicted_family_label": row_a["predicted_family_label"],
            "selection_type": row_a["selection_type"],
            "is_correct": int(row_a["is_correct"]),
            "frame_idx_a": int(row_a["frame_index_local"]),
            "frame_idx_b": int(row_b["frame_index_local"]),
            "pearson_similarity": pearson_val,
            "cosine_similarity": cosine_val,
            "l1_similarity": l1_val,
        })

        pearsons.append(pearson_val)
        cosines.append(cosine_val)
        l1s.append(l1_val)

    video_summary_records.append({
        "sample_id": sample_id,
        "dataset": sub_df.iloc[0].get("dataset", None),
        "true_family_label": sub_df.iloc[0]["true_family_label"],
        "predicted_family_label": sub_df.iloc[0]["predicted_family_label"],
        "selection_type": sub_df.iloc[0]["selection_type"],
        "is_correct": int(sub_df.iloc[0]["is_correct"]),
        "n_adjacent_pairs": int(len(sub_df) - 1),
        "mean_pearson_similarity": float(np.nanmean(pearsons)),
        "std_pearson_similarity": float(np.nanstd(pearsons)),
        "mean_cosine_similarity": float(np.nanmean(cosines)),
        "std_cosine_similarity": float(np.nanstd(cosines)),
        "mean_l1_similarity": float(np.nanmean(l1s)),
        "std_l1_similarity": float(np.nanstd(l1s)),
    })

framepair_df = pd.DataFrame(framepair_records)
video_summary_df = pd.DataFrame(video_summary_records)

framepair_df.to_csv(FRAMEPAIR_CSV_MC, index=False)
video_summary_df.to_csv(VIDEO_SUMMARY_CSV_MC, index=False)


# Aggregate summaries

overall_summary = {
    "n_samples_analyzed": int(video_summary_df["sample_id"].nunique()),
    "n_adjacent_pairs_analyzed": int(len(framepair_df)),
    "overall_mean_pearson_similarity": float(video_summary_df["mean_pearson_similarity"].mean()),
    "overall_mean_cosine_similarity": float(video_summary_df["mean_cosine_similarity"].mean()),
    "overall_mean_l1_similarity": float(video_summary_df["mean_l1_similarity"].mean()),
}

by_true_class = (
    video_summary_df.groupby("true_family_label")[[
        "mean_pearson_similarity",
        "mean_cosine_similarity",
        "mean_l1_similarity"
    ]]
    .mean()
    .reset_index()
)

by_correctness = (
    video_summary_df.groupby("is_correct")[[
        "mean_pearson_similarity",
        "mean_cosine_similarity",
        "mean_l1_similarity"
    ]]
    .mean()
    .reset_index()
)

by_selection_type = (
    video_summary_df.groupby("selection_type")[[
        "mean_pearson_similarity",
        "mean_cosine_similarity",
        "mean_l1_similarity"
    ]]
    .mean()
    .reset_index()
)

summary_json = {
    "overall": overall_summary,
    "by_true_class": by_true_class.to_dict(orient="records"),
    "by_correctness": by_correctness.to_dict(orient="records"),
    "by_selection_type": by_selection_type.to_dict(orient="records"),
}

with open(TEMPORAL_SUMMARY_JSON_MC, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)


# Print summary

print("\nTemporal consistency summary:")
print("  Samples analyzed              :", overall_summary["n_samples_analyzed"])
print("  Adjacent frame pairs analyzed :", overall_summary["n_adjacent_pairs_analyzed"])

print("\nOverall:")
print("  Mean Pearson similarity       :", f"{overall_summary['overall_mean_pearson_similarity']:.4f}")
print("  Mean Cosine similarity        :", f"{overall_summary['overall_mean_cosine_similarity']:.4f}")
print("  Mean L1 similarity            :", f"{overall_summary['overall_mean_l1_similarity']:.4f}")

print("\nBy true class:")
display(by_true_class)

print("\nBy correctness:")
display(by_correctness)

print("\nBy selection type:")
display(by_selection_type)

print("\nPer-video summary preview:")
display(video_summary_df.head(20))

print("\nSaved:")
print("  Framepair CSV        :", FRAMEPAIR_CSV_MC)
print("  Video summary CSV    :", VIDEO_SUMMARY_CSV_MC)
print("  Summary JSON         :", TEMPORAL_SUMMARY_JSON_MC)